# Subetapa 4.4 — entrenamiento y evaluación por grupos

**SYNTHETIC / DEMO / ACADEMIC DATASET**. Solo título y descripción; sin conexión a tickets operativos.

El notebook reutiliza la preparación 4.3. 160 registros de desarrollo; 40 de prueba final protegida. Baseline, regresión logística y SVM lineal; protocolo y particiones congelados antes de entrenar.

Ejecuta desde un entorno Colab nuevo con CPU. La prueba final está **desactivada inicialmente**: primero observa comparación/decisión, después activa solo la celda correspondiente. No cambies parámetros ni selecciones candidatos usando esa prueba. No exporta un modelo; 4.5 requiere autorización.

Guía: [COLAB-4.4.md](https://github.com/C-David28/essalud-ticket-system/blob/main/ml/docs/COLAB-4.4.md).


## 1. Herramientas de preparación

Código reproducible incluido desde `ml/colab/preparation.py`. Solo usa la biblioteca estándar. Las descargas se limitan a cinco archivos permitidos; no se descarga ni ejecuta código desde el dataset. Los ZIP se leen en memoria, sin extraerlos.

No necesitas modificar esta celda.


In [ ]:
"""Stage 4.3 only: immutable inputs, safe loading, dataset and environment checks.

Standard library only. No model fitting, predictions, split creation or database access.
This source is embedded into the notebook by build-colab-notebook.mjs.
"""
from collections import Counter
from hashlib import sha256
from io import BytesIO
import json
from pathlib import Path, PurePosixPath
import re
import stat
import subprocess
import sys
from urllib.error import HTTPError, URLError
from urllib.request import Request, build_opener, HTTPRedirectHandler
import zipfile

REPOSITORY = "C-David28/essalud-ticket-system"
DATASET_COMMIT = "9d3103991911b1b06b9faca6dce779013dae3036"
DATASET_VERSION = "tickets-category-dataset-v1.0.0"
PREPARATION_VERSION = "stage-4.3-preparation-v1"
INSTALLER_VERSION = "25.1.1"
DATASET_ROOT = "ml/datasets/v1.0.0/"
SOURCE = "SYNTHETIC / DEMO / ACADEMIC DATASET"
CATEGORIES = ["SOPORTE", "REDES", "INFRAESTRUCTURA", "BIOMEDICO"]
EXPECTED_HASHES = {
    DATASET_ROOT + "incidents.jsonl": "2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69",
    DATASET_ROOT + "families.json": "0f27342915a770acd019a31b7685c8a2eea8c2de7616a38ca2ac8c1f079078b7",
    "ml/docs/LABELING-GUIDE.md": "7cb0ee07c57a6fb4a857bbde3ba8fc6f6b1e41d911d81682c62b98f316899cc9",
    DATASET_ROOT + "manifest.json": "aa698ab7e4480e2a0f51085b5a5138b2267eb12a021eb08c590d58a7d11fdb61",
    "apps/api/src/domain/ticket.ts": "8cfa93c9f1e9a8a7106c46118671d38c63cfce2041e514210430d0bf7560ce3a",
}
ALLOWED_FILES = (
    DATASET_ROOT + "incidents.jsonl",
    DATASET_ROOT + "families.json",
    DATASET_ROOT + "manifest.json",
    "ml/docs/LABELING-GUIDE.md",
    "apps/api/src/domain/ticket.ts",
)
MAX_FILE_BYTES = 1_000_000
MAX_ARCHIVE_BYTES = 3_000_000


def require(condition, code):
    if not condition:
        raise ValueError(code)


def canonical_hash(payload):
    return sha256(payload.decode("utf-8").replace("\r\n", "\n").encode("utf-8")).hexdigest()


def parse_json(payload, code):
    try:
        return json.loads(payload.decode("utf-8"))
    except (ValueError, UnicodeError):
        raise ValueError(code) from None


class NoRedirect(HTTPRedirectHandler):
    # Never forward a private Authorization header to an unexpected host.
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


def load_github(token=None, opener=None):
    """Read only the allowlisted files at a fixed SHA; token stays in memory/header."""
    request_opener = opener or build_opener(NoRedirect())
    payloads = {}
    for name in ALLOWED_FILES:
        url = f"https://api.github.com/repos/{REPOSITORY}/contents/{name}?ref={DATASET_COMMIT}"
        headers = {"Accept": "application/vnd.github.raw+json",
                   "X-GitHub-Api-Version": "2022-11-28", "User-Agent": "essalud-ml-stage-4.3"}
        if token:
            headers["Authorization"] = "Bearer " + token
        try:
            with request_opener.open(Request(url, headers=headers), timeout=30) as response:
                payload = response.read(MAX_FILE_BYTES + 1)
        except HTTPError as error:
            raise RuntimeError(f"GITHUB_READ_FAILED_HTTP_{error.code}: verify commit, access or use ZIP upload") from None
        except (URLError, OSError):
            raise RuntimeError("GITHUB_READ_FAILED: network unavailable; use verified ZIP upload") from None
        require(len(payload) <= MAX_FILE_BYTES, "GITHUB_FILE_TOO_LARGE")
        payloads[name] = payload
    return payloads


def load_archive(payload):
    """Read an allowlisted git archive in memory. Never extract it to the filesystem."""
    require(isinstance(payload, bytes) and len(payload) <= MAX_ARCHIVE_BYTES, "ARCHIVE_TOO_LARGE")
    result, seen, total = {}, set(), 0
    try:
        with zipfile.ZipFile(BytesIO(payload)) as archive:
            require(len(archive.infolist()) <= 25, "ARCHIVE_TOO_MANY_ENTRIES")
            for entry in archive.infolist():
                name = entry.filename
                original = entry.orig_filename
                parts = PurePosixPath(name).parts
                require(original == name and name and "\\" not in original and not name.startswith("/") and
                        ":" not in name and ".." not in parts and name not in seen, "UNSAFE_ARCHIVE_PATH")
                seen.add(name)
                require(not stat.S_ISLNK(entry.external_attr >> 16), "ARCHIVE_SYMLINK")
                if entry.is_dir():
                    require(any(item.startswith(name) for item in ALLOWED_FILES), "UNEXPECTED_ARCHIVE_DIRECTORY")
                    continue
                require(name in ALLOWED_FILES, "UNEXPECTED_ARCHIVE_FILE")
                total += entry.file_size
                require(entry.file_size <= MAX_FILE_BYTES and total <= MAX_ARCHIVE_BYTES, "ARCHIVE_EXPANDED_TOO_LARGE")
                result[name] = archive.read(entry)
    except (zipfile.BadZipFile, RuntimeError):
        raise ValueError("INVALID_OR_ENCRYPTED_ARCHIVE") from None
    require(set(result) == set(ALLOWED_FILES), "ARCHIVE_MISSING_FILE")
    return result


def load_local_snapshot(repository_root):
    """Offline smoke tests only; same allowlist and hashes as GitHub/ZIP."""
    root = Path(repository_root)
    return {name: (root / name).read_bytes() for name in ALLOWED_FILES}


def verify_dataset(payloads):
    require(set(payloads) == set(ALLOWED_FILES), "INPUT_FILE_SET_MISMATCH")
    require(all(isinstance(value, bytes) and len(value) <= MAX_FILE_BYTES for value in payloads.values()), "INVALID_PAYLOAD")
    for name, digest in EXPECTED_HASHES.items():
        require(canonical_hash(payloads[name]) == digest, "IMMUTABLE_DATASET_HASH_MISMATCH")
    manifest = parse_json(payloads[DATASET_ROOT + "manifest.json"], "INVALID_MANIFEST_JSON")
    require(isinstance(manifest, dict), "INVALID_MANIFEST")
    require(manifest.get("dataset_version") == DATASET_VERSION and manifest.get("schema_version") == 1 and
            manifest.get("source") == SOURCE and manifest.get("hash_algorithm") == "SHA-256-UTF8-LF", "RELEASE_MISMATCH")
    require(manifest.get("categories") == CATEGORIES and manifest.get("distribution") == dict.fromkeys(CATEGORIES, 50), "CATEGORY_MANIFEST_MISMATCH")
    require(manifest.get("record_count") == 200 and manifest.get("family_count") == 40 and
            manifest.get("leakage_group_count") == 37 and manifest.get("records_per_family") == 5, "COUNT_MANIFEST_MISMATCH")
    expected_manifest_hashes = {"incidents.jsonl": EXPECTED_HASHES[DATASET_ROOT + "incidents.jsonl"],
                               "families.json": EXPECTED_HASHES[DATASET_ROOT + "families.json"],
                               "../../docs/LABELING-GUIDE.md": EXPECTED_HASHES["ml/docs/LABELING-GUIDE.md"]}
    require(manifest.get("files") == expected_manifest_hashes and manifest.get("training_status") == "NOT_STARTED" and
            manifest.get("splits_status") == "NOT_CREATED_STAGE_4_3", "FROZEN_MANIFEST_MISMATCH")
    match = re.search(r"export const CATEGORIES\s*=\s*\[([^\]]+)\]", payloads["apps/api/src/domain/ticket.ts"].decode("utf-8"))
    require(match is not None and re.findall(r"['\"]([^'\"]+)['\"]", match.group(1)) == CATEGORIES, "DOMAIN_CATEGORY_MISMATCH")
    catalog = parse_json(payloads[DATASET_ROOT + "families.json"], "INVALID_FAMILIES_JSON")
    require(isinstance(catalog, dict) and catalog.get("schema_version") == 1 and
            isinstance(catalog.get("families"), list), "INVALID_FAMILY_CATALOG")
    families = {row["scenario_family_id"]: row for row in catalog["families"]}
    require(len(families) == len(catalog["families"]) == 40, "INVALID_FAMILY_COUNT")
    lines = payloads[DATASET_ROOT + "incidents.jsonl"].decode("utf-8").splitlines()
    records = [parse_json(line.encode("utf-8"), "INVALID_RECORD_JSON") for line in lines]
    require(len(records) == 200, "INVALID_RECORD_COUNT")
    fields = {"record_id", "titulo", "descripcion", "categoria", "scenario_family_id", "leakage_group_id",
              "is_synthetic", "source", "dataset_version", "label_status", "label_rationale",
              "writing_style", "length_band", "is_boundary_case"}
    ids, inputs = set(), set()
    for row in records:
        require(isinstance(row, dict) and set(row) == fields, "INVALID_RECORD_FIELDS")
        require(isinstance(row["record_id"], str) and re.fullmatch(r"ML-\d{4}", row["record_id"]) and
                row["record_id"] not in ids, "INVALID_RECORD_ID")
        ids.add(row["record_id"])
        require(row["is_synthetic"] is True and row["source"] == SOURCE and row["dataset_version"] == DATASET_VERSION and
                row["label_status"] == "APPROVED", "INVALID_RECORD_PROVENANCE")
        require(row["categoria"] in CATEGORIES, "INVALID_CATEGORY")
        family = families.get(row["scenario_family_id"])
        require(family is not None and row["categoria"] == family["categoria"] and
                row["leakage_group_id"] == family["leakage_group_id"], "FAMILY_GROUP_MISMATCH")
        require(isinstance(row["titulo"], str) and 5 <= len(row["titulo"]) <= 200 and
                isinstance(row["descripcion"], str) and 10 <= len(row["descripcion"]) <= 5000, "INVALID_TEXT")
        text = (row["titulo"].strip(), row["descripcion"].strip())
        require(text not in inputs, "DUPLICATE_TEXT")
        inputs.add(text)
    distribution = dict(Counter(row["categoria"] for row in records))
    require(distribution == dict.fromkeys(CATEGORIES, 50), "UNBALANCED_DATASET")
    require(all(count == 5 for count in Counter(row["scenario_family_id"] for row in records).values()), "INVALID_FAMILY_SIZE")
    groups = len({row["leakage_group_id"] for row in records})
    require(groups == 37, "INVALID_GROUP_COUNT")
    report = {"dataset_version": DATASET_VERSION, "dataset_commit": DATASET_COMMIT,
              "dataset_sha256": EXPECTED_HASHES[DATASET_ROOT + "incidents.jsonl"],
              "records": len(records), "distribution": distribution, "families": len(families),
              "leakage_groups": groups, "is_synthetic": True, "training_started": False,
              "splits_created": False, "status": "DATASET_V1_VERIFIED"}
    return records, report


def parse_requirements(text):
    pins = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = re.fullmatch(r"([a-zA-Z0-9-]+)==([0-9]+(?:\.[0-9]+)+(?:\.post[0-9]+)?)", line)
        require(match is not None and match.group(1) not in pins, "INVALID_DEPENDENCY_PIN")
        pins[match.group(1)] = match.group(2)
    require(len(pins) == 18, "INCOMPLETE_DEPENDENCY_PINS")
    return pins


def check_python():
    require((3, 11) <= sys.version_info[:2] < (3, 14), "PYTHON_UNSUPPORTED: use Python 3.11, 3.12 or 3.13")


def prepare_environment(workspace, requirements_text, install=True):
    """Use a project-local venv so Colab's system packages remain untouched."""
    check_python()
    parse_requirements(requirements_text)
    workspace = Path(workspace).resolve()
    workspace.mkdir(parents=True, exist_ok=True)
    environment = workspace / ".venv"
    python = environment / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    if not python.exists():
        # Colab images may lack ensurepip; host pip manages this empty venv directly.
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(environment)], check=True, timeout=120)
    requirements_file = workspace / "requirements-stage-4.3.txt"
    requirements_file.write_text(requirements_text, encoding="utf-8")
    if install:
        subprocess.run([sys.executable, "-m", "pip", "--isolated", "--python", str(python), "install", "--disable-pip-version-check",
                        "--only-binary=:all:", "--index-url", "https://pypi.org/simple",
                        f"pip=={INSTALLER_VERSION}", "-r", str(requirements_file)], check=True, timeout=600)
    return python


def verify_environment(python, requirements_text):
    pins = parse_requirements(requirements_text)
    code = """import importlib.metadata as m, json, sys
import numpy, scipy, pandas, sklearn, matplotlib
print(json.dumps({'python':sys.version.split()[0], 'pip_tool_version':m.version('pip'), 'packages':{name:m.version(name) for name in json.loads(sys.argv[1])}}))
"""
    result = subprocess.run([str(python), "-c", code, json.dumps(list(pins))],
                            check=True, capture_output=True, text=True, timeout=120)
    report = json.loads(result.stdout)
    require(report["packages"] == pins, "INSTALLED_DEPENDENCY_VERSION_MISMATCH")
    require(report["pip_tool_version"] == INSTALLER_VERSION, "INSTALLER_VERSION_MISMATCH")
    require((3, 11) <= tuple(map(int, report["python"].split(".")[:2])) < (3, 14), "ENVIRONMENT_PYTHON_UNSUPPORTED")
    subprocess.run([str(python), "-m", "pip", "check"], check=True, timeout=60)
    return report


def readiness_report(dataset_report, environment_report, requirements_text, source_mode, preparation_sha256):
    require(re.fullmatch(r"[0-9a-f]{64}", preparation_sha256) is not None, "INVALID_PREPARATION_HASH")
    return {"stage": "4.3", "status": "READY_FOR_STAGE_4_4", "source_mode": source_mode,
            "repository": REPOSITORY, "dataset": dataset_report, "environment": environment_report,
            "preparation_version": PREPARATION_VERSION, "preparation_source_sha256_utf8_lf": preparation_sha256,
            "requirements_sha256_utf8_lf": canonical_hash(requirements_text.encode("utf-8")),
            "training_started": False, "model_selected": False, "operational_database_access": False}


## 2. Configuración

Para el repositorio público actual deja **github_public**. Alternativas:

- **upload**: carga el ZIP exportado con `npm run ml:colab:export`; recomendado si el repositorio es privado, sin token.
- **github_secret**: opcional; solo para lectura privada automática. Colab Secrets debe contener `GITHUB_READ_TOKEN` con acceso autorizado a este notebook. Nunca escribas el valor aquí.
- **local_snapshot**: reservado a pruebas locales/CI; no usar en Colab.

Las dependencias se instalan dentro de `WORKSPACE/.venv`. El Python del experimento queda en `experiment_python`: evita usar los paquetes globales de Colab para el experimento. No se necesita GPU ni reiniciar el kernel por cambios de NumPy global.


In [ ]:
SOURCE_MODE = "github_public"
LOCAL_REPOSITORY = None  # Solo pruebas locales; no configurar en Colab.
WORKSPACE = Path("/content/essalud-ml-stage-4.4")
INSTALL_DEPENDENCIES = True
PREPARATION_SOURCE_SHA256 = "e2ac1f6e1f2eb0da013e253869899ab4800c0e007c3a959d4c459a9dcaf06c98"
REQUIREMENTS_TEXT = "# Scientific environment for stages 4.3-4.4; Python 3.11-3.13; CPU only.\n# All runtime dependencies are pinned; pip itself is only an installation tool.\nnumpy==2.2.6\nscipy==1.15.3\npandas==2.2.3\nscikit-learn==1.6.1\nmatplotlib==3.10.3\njoblib==1.5.1\nthreadpoolctl==3.6.0\ncontourpy==1.3.2\ncycler==0.12.1\nfonttools==4.58.0\nkiwisolver==1.4.8\npackaging==25.0\npillow==11.2.1\npyparsing==3.2.3\npython-dateutil==2.9.0.post0\npytz==2025.2\nsix==1.17.0\ntzdata==2025.2\n"
print("Dataset:", DATASET_VERSION)
print("Commit de datos:", DATASET_COMMIT)
print("Origen:", SOURCE_MODE)


## 3. Preparar y comprobar dependencias

Python compatible: **3.11, 3.12 o 3.13**. Se fijan 18 dependencias de cálculo, tablas y gráficos para el experimento futuro; instalarlas no selecciona ningún algoritmo.

La primera ejecución puede tardar por las descargas de PyPI. Esta celda comprueba versiones reales, imports y `pip check`. Los paquetes preinstalados de Colab permanecen intactos. La versión exacta de Python se registra; Colab puede cambiar su imagen base.


In [ ]:
experiment_python = prepare_environment(WORKSPACE, REQUIREMENTS_TEXT, install=INSTALL_DEPENDENCIES)
environment_report = verify_environment(experiment_python, REQUIREMENTS_TEXT)
print(json.dumps(environment_report, ensure_ascii=False, indent=2))


## 4. Cargar los archivos versionados

**Público:** lectura sin autenticación de un commit inmutable. Si GitHub devuelve 404, comprueba que publicaste el commit de Dataset V1; no reemplaces su SHA por main.

**Privado:** abrir un notebook con tu sesión de GitHub no autentica automáticamente las descargas de la máquina de Colab. Usa el ZIP permitido o, solo si lo necesitas, un secreto de lectura.

No cargues un ZIP del directorio completo de tu proyecto, .env, backups ni datos de producción.


In [ ]:
def load_private_from_secret():
    try:
        from google.colab import userdata
        secret = userdata.get("GITHUB_READ_TOKEN")
    except Exception:
        raise RuntimeError("Autoriza GITHUB_READ_TOKEN en Colab Secrets o usa upload sin token.") from None
    try:
        return load_github(token=secret)
    finally:
        secret = None

if SOURCE_MODE == "github_public":
    payloads = load_github()
elif SOURCE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    require(len(uploaded) == 1, "Sube solamente dataset-v1-colab.zip")
    try:
        payloads = load_archive(bytes(next(iter(uploaded.values()))))
    finally:
        uploaded.clear()
elif SOURCE_MODE == "github_secret":
    payloads = load_private_from_secret()
elif SOURCE_MODE == "local_snapshot":
    require(LOCAL_REPOSITORY is not None, "LOCAL_REPOSITORY_REQUIRED")
    payloads = load_local_snapshot(LOCAL_REPOSITORY)
else:
    raise ValueError("SOURCE_MODE_INVALID")
print("Archivos permitidos cargados:", len(payloads))


## 5. Verificar Dataset V1 antes del experimento

Se verifican cinco hashes, 200 registros, cuatro categorías con 50 cada una, 40 familias y 37 grupos. El manifiesto V1 conserva el estado de su publicación previa, sin entrenamiento. La actividad de esta ejecución se registra en los reportes 4.4, sin modificar aquella versión.

Las columnas auxiliares no entran al clasificador. Las particiones ya congeladas se comprobarán en la celda siguiente.


In [ ]:
records, dataset_report = verify_dataset(payloads)
MODEL_INPUT_COLUMNS = ("titulo", "descripcion")
print(json.dumps({key: value for key, value in dataset_report.items() if key not in {"training_started", "splits_created"}}, ensure_ascii=False, indent=2))
print("Únicas entradas del modelo:", MODEL_INPUT_COLUMNS)


## 6. Protocolo y particiones congelados

Seed 42; holdout exacto 10 por categoría por grupos/etiquetas, sin mirar resultados. CV de 4 folds agrupados; algunas validaciones tienen tamaños diferentes. TF-IDF de palabras 1–2 y caracteres 3–5, ajustado dentro de cada entrenamiento.

Baseline mayoría; LogisticRegression C=1/4 y LinearSVC C=1/4. Selección solo con Macro F1 medio CV, recall OOF de cada clase y mejora frente al baseline. Objetivos prospectivos: 0.70, 0.60 y 0.10. Diferencias dentro de la dispersión favorecen el algoritmo/preferencia C fijados. Si no pasan todos los criterios, se documenta y no se promueve.


In [ ]:
EXPERIMENT_SOURCE = "\"\"\"Stage 4.4: frozen grouped evaluation. No database, HTTP service or model export.\n\nOnly title and description enter estimators. Final test is a separate explicit phase.\n\"\"\"\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom hashlib import sha256\nimport argparse\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\n\n# Colab writes preparation.py beside this file; the repository reuses the 4.3 helper.\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"colab\"))\nimport preparation as prep\n\nVERSION = \"ticket-category-experiment-v1.0.0\"\nLABELS = prep.CATEGORIES\n\n\ndef digest(value):\n    return sha256(json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\",\", \":\")).encode()).hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\"utf-8\"))\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + \"\\n\", encoding=\"utf-8\")\n\n\ndef text_input(row):\n    # Never pass category, rationale, site, ID, family or group to the vectorizers.\n    return row[\"titulo\"].strip() + \"\\n\" + row[\"descripcion\"].strip()\n\n\ndef group_holdout(records, seed=42):\n    \"\"\"Exact 10/class selection using labels/groups only, before any fit or text scoring.\"\"\"\n    counts = {}\n    for row in records:\n        counts.setdefault(row[\"leakage_group_id\"], Counter())[row[\"categoria\"]] += 1\n    ordered = sorted(counts, key=lambda group: sha256(f\"{seed}|{group}\".encode()).hexdigest())\n    states = {(0, 0, 0, 0): ()}\n    target = (10, 10, 10, 10)\n    for group in ordered:\n        vector = tuple(counts[group][label] for label in LABELS)\n        for current, chosen in list(states.items()):\n            new = tuple(a + b for a, b in zip(current, vector))\n            if all(a <= b for a, b in zip(new, target)):\n                states.setdefault(new, chosen + (group,))\n    prep.require(target in states, \"NO_BALANCED_GROUP_HOLDOUT\")\n    return set(states[target])\n\n\ndef validate_protocol(protocol):\n    prep.require(protocol[\"experiment_version\"] == VERSION and protocol[\"dataset_sha256\"] == prep.EXPECTED_HASHES[prep.DATASET_ROOT + \"incidents.jsonl\"], \"PROTOCOL_DATASET_MISMATCH\")\n    prep.require(protocol[\"dataset_commit\"] == prep.DATASET_COMMIT and protocol[\"input_columns\"] == [\"titulo\", \"descripcion\"], \"PROTOCOL_INPUT_MISMATCH\")\n    prep.require(protocol[\"persist_model\"] is False and protocol[\"institutional_validation\"] is False, \"PROTOCOL_SCOPE_MISMATCH\")\n    prep.require(protocol[\"validation\"][\"folds\"] == 4 and protocol[\"validation\"][\"group_column\"] == \"leakage_group_id\", \"PROTOCOL_GROUP_MISMATCH\")\n\n\ndef freeze_splits(records, protocol):\n    import numpy as np\n    from sklearn.model_selection import StratifiedGroupKFold\n    validate_protocol(protocol)\n    held_groups = group_holdout(records, protocol[\"seed\"])\n    development = [r for r in records if r[\"leakage_group_id\"] not in held_groups]\n    test = [r for r in records if r[\"leakage_group_id\"] in held_groups]\n    y = np.array([r[\"categoria\"] for r in development])\n    groups = np.array([r[\"leakage_group_id\"] for r in development])\n    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=protocol[\"seed\"])\n    folds = [{\"fold\": number + 1,\n              \"training_ids\": [development[i][\"record_id\"] for i in train],\n              \"validation_ids\": [development[i][\"record_id\"] for i in valid]}\n             for number, (train, valid) in enumerate(cv.split(np.zeros(len(y)), y, groups))]\n    result = {\"schema_version\": 1, \"experiment_version\": VERSION, \"dataset_sha256\": protocol[\"dataset_sha256\"],\n              \"protocol_sha256_canonical_json\": digest(protocol), \"seed\": protocol[\"seed\"],\n              \"method\": protocol[\"holdout\"][\"method\"],\n              \"development_ids\": [r[\"record_id\"] for r in development],\n              \"final_test_ids\": [r[\"record_id\"] for r in test], \"folds\": folds}\n    validate_splits(records, protocol, result)\n    return result\n\n\ndef validate_splits(records, protocol, splits):\n    validate_protocol(protocol)\n    prep.require(splits[\"protocol_sha256_canonical_json\"] == digest(protocol) and splits[\"dataset_sha256\"] == protocol[\"dataset_sha256\"], \"SPLIT_PROTOCOL_MISMATCH\")\n    by_id = {r[\"record_id\"]: r for r in records}\n    all_ids = set(by_id)\n    dev, test = set(splits[\"development_ids\"]), set(splits[\"final_test_ids\"])\n    prep.require(len(dev) == len(splits[\"development_ids\"]) == 160 and len(test) == len(splits[\"final_test_ids\"]) == 40, \"SPLIT_SIZE_MISMATCH\")\n    prep.require(not dev & test and dev | test == all_ids, \"SPLIT_COVERAGE_MISMATCH\")\n    groups = lambda ids: {by_id[i][\"leakage_group_id\"] for i in ids}\n    prep.require(not groups(dev) & groups(test), \"HOLDOUT_GROUP_LEAKAGE\")\n    prep.require(Counter(by_id[i][\"categoria\"] for i in test) == dict.fromkeys(LABELS, 10), \"HOLDOUT_CLASS_MISMATCH\")\n    validation_ids = []\n    prep.require(len(splits[\"folds\"]) == 4, \"CV_FOLD_COUNT_MISMATCH\")\n    for fold in splits[\"folds\"]:\n        train, valid = set(fold[\"training_ids\"]), set(fold[\"validation_ids\"])\n        prep.require(len(train) == len(fold[\"training_ids\"]) and len(valid) == len(fold[\"validation_ids\"]), \"CV_DUPLICATE_IDS\")\n        prep.require(not train & valid and train | valid == dev, \"CV_COVERAGE_MISMATCH\")\n        prep.require(not groups(train) & groups(valid), \"CV_GROUP_LEAKAGE\")\n        prep.require({by_id[i][\"categoria\"] for i in train} == set(LABELS) and {by_id[i][\"categoria\"] for i in valid} == set(LABELS), \"CV_CLASS_MISSING\")\n        validation_ids += fold[\"validation_ids\"]\n    prep.require(Counter(validation_ids) == Counter({i: 1 for i in dev}), \"OOF_COVERAGE_MISMATCH\")\n    return {\"development\": {\"records\": len(dev), \"groups\": len(groups(dev)), \"distribution\": dict(Counter(by_id[i][\"categoria\"] for i in dev))},\n            \"final_test\": {\"records\": len(test), \"groups\": len(groups(test)), \"distribution\": dict(Counter(by_id[i][\"categoria\"] for i in test))},\n            \"folds\": [{\"fold\": f[\"fold\"], \"training\": len(f[\"training_ids\"]), \"validation\": len(f[\"validation_ids\"]),\n                       \"validation_distribution\": dict(Counter(by_id[i][\"categoria\"] for i in f[\"validation_ids\"]))} for f in splits[\"folds\"]]}\n\n\ndef make_estimator(spec, protocol):\n    from sklearn.dummy import DummyClassifier\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.pipeline import FeatureUnion, Pipeline\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.svm import LinearSVC\n    if spec[\"algorithm\"] == \"DummyClassifier\":\n        return DummyClassifier(strategy=spec[\"strategy\"], random_state=protocol[\"seed\"])\n    vector = protocol[\"vectorizer\"]\n    shared = {key: vector[key] for key in [\"lowercase\", \"strip_accents\", \"sublinear_tf\", \"min_df\"]}\n    features = FeatureUnion([\n        (\"word\", TfidfVectorizer(analyzer=\"word\", ngram_range=tuple(vector[\"word_ngram_range\"]), stop_words=None, **shared)),\n        (\"char\", TfidfVectorizer(analyzer=\"char_wb\", ngram_range=tuple(vector[\"char_ngram_range\"]), **shared))],\n        transformer_weights={\"word\": vector[\"view_weights\"][0], \"char\": vector[\"view_weights\"][1]})\n    if spec[\"algorithm\"] == \"LogisticRegression\":\n        classifier = LogisticRegression(C=spec[\"C\"], solver=spec[\"solver\"], max_iter=spec[\"max_iter\"], random_state=protocol[\"seed\"])\n    else:\n        prep.require(spec[\"algorithm\"] == \"LinearSVC\", \"UNSUPPORTED_CANDIDATE\")\n        classifier = LinearSVC(C=spec[\"C\"], dual=spec[\"dual\"], max_iter=spec[\"max_iter\"], random_state=protocol[\"seed\"])\n    return Pipeline([(\"features\", features), (\"classifier\", classifier)])\n\n\ndef metrics(y, predictions):\n    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\n    report = classification_report(y, predictions, labels=LABELS, output_dict=True, zero_division=0)\n    matrix = confusion_matrix(y, predictions, labels=LABELS)\n    normalized = confusion_matrix(y, predictions, labels=LABELS, normalize=\"true\")\n    return {\"accuracy\": float(accuracy_score(y, predictions)), \"macro_f1\": float(f1_score(y, predictions, labels=LABELS, average=\"macro\", zero_division=0)),\n            \"weighted_f1\": float(f1_score(y, predictions, labels=LABELS, average=\"weighted\", zero_division=0)),\n            \"classification_report\": report, \"labels\": LABELS, \"confusion_matrix\": matrix.tolist(), \"confusion_matrix_normalized\": normalized.tolist()}\n\n\ndef meets_gates(macro, per_class, baseline_macro, protocol):\n    gates = protocol[\"selection\"]\n    return macro >= gates[\"minimum_macro_f1\"] and all(per_class[label][\"recall\"] >= gates[\"minimum_recall_per_category\"] for label in LABELS) and macro - baseline_macro >= gates[\"minimum_macro_f1_gain_over_baseline\"]\n\n\ndef choose_candidate(comparison, protocol):\n    \"\"\"This function receives development metrics only, never a final-test report.\"\"\"\n    rows = comparison[\"candidates\"]\n    baseline = next(r for r in rows if r[\"algorithm\"] == \"DummyClassifier\")\n    candidates = [r for r in rows if r[\"algorithm\"] != \"DummyClassifier\"]\n    qualified = [r for r in candidates if meets_gates(r[\"mean_macro_f1\"], r[\"oof_metrics\"][\"classification_report\"], baseline[\"mean_macro_f1\"], protocol)]\n    pool = qualified or candidates\n    best = max(pool, key=lambda r: (r[\"mean_macro_f1\"], r[\"id\"]))\n    nearby = [r for r in pool if best[\"mean_macro_f1\"] - r[\"mean_macro_f1\"] <= max(best[\"std_macro_f1\"], r[\"std_macro_f1\"])]\n    ranks = {name: index for index, name in enumerate(protocol[\"selection\"][\"tie_preference\"])}\n    chosen = min(nearby, key=lambda r: (ranks[r[\"algorithm\"]], r[\"spec\"][\"C\"], -r[\"mean_macro_f1\"]))\n    return {\"candidate_id\": chosen[\"id\"], \"candidate_spec\": chosen[\"spec\"], \"selection_source\": \"development_grouped_cv_only\",\n            \"cv_acceptance_passed\": bool(qualified), \"best_mean_candidate_id\": best[\"id\"],\n            \"near_tie_candidates\": [r[\"id\"] for r in nearby], \"chosen_mean_macro_f1\": chosen[\"mean_macro_f1\"],\n            \"chosen_std_macro_f1\": chosen[\"std_macro_f1\"], \"baseline_mean_macro_f1\": baseline[\"mean_macro_f1\"],\n            \"reason\": \"Eligibility gates first; CV mean/variability and predeclared algorithm/lower-C preference\" if qualified else \"No candidate passed all gates; evaluate leading development candidate, never promote on final-test performance\"}\n\n\ndef score_outputs(estimator, texts, predicted, spec):\n    import numpy as np\n    if spec[\"algorithm\"] == \"LogisticRegression\":\n        probabilities = estimator.predict_proba(texts)\n        return np.max(probabilities, axis=1).tolist(), \"uncalibrated_probability\"\n    if spec[\"algorithm\"] == \"LinearSVC\":\n        decision = estimator.decision_function(texts)\n        return np.max(decision, axis=1).tolist(), \"raw_decision_margin_not_probability\"\n    return [None] * len(predicted), \"baseline_no_confidence\"\n\n\ndef write_csv(path, rows, fieldnames=None):\n    if not rows and not fieldnames:\n        return\n    with Path(path).open(\"w\", encoding=\"utf-8\", newline=\"\") as file:\n        writer = csv.DictWriter(file, fieldnames=fieldnames or list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef compare(records, protocol, splits, output, provenance):\n    import numpy as np\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    prep.require(not (output / \"selection.json\").exists(), \"COMPARISON_EXISTS: use existing report; do not overwrite a frozen decision\")\n    # Exclude test rows before preprocessing, fitting, scoring or selection.\n    selected_ids = set(splits[\"development_ids\"])\n    development = {r[\"record_id\"]: r for r in records if r[\"record_id\"] in selected_ids}\n    rows, oof_rows = [], []\n    for spec in protocol[\"candidates\"]:\n        folds, predictions, scores, fit_times, infer_times = [], {}, {}, [], []\n        score_kind = None\n        for fold in splits[\"folds\"]:\n            train = [development[i] for i in fold[\"training_ids\"]]\n            valid = [development[i] for i in fold[\"validation_ids\"]]\n            estimator = make_estimator(spec, protocol)\n            start = time.perf_counter()\n            estimator.fit([text_input(r) for r in train], [r[\"categoria\"] for r in train])\n            fit_times.append(time.perf_counter() - start)\n            texts = [text_input(r) for r in valid]\n            start = time.perf_counter()\n            pred = estimator.predict(texts).tolist()\n            infer_times.append((time.perf_counter() - start) / len(valid))\n            values, score_kind = score_outputs(estimator, texts, pred, spec)\n            predictions.update(zip(fold[\"validation_ids\"], pred))\n            scores.update(zip(fold[\"validation_ids\"], values))\n            folds.append({\"fold\": fold[\"fold\"], **metrics([r[\"categoria\"] for r in valid], pred)})\n        ordered = splits[\"development_ids\"]\n        y = [development[i][\"categoria\"] for i in ordered]\n        pred = [predictions[i] for i in ordered]\n        row = {\"id\": spec[\"id\"], \"algorithm\": spec[\"algorithm\"], \"spec\": spec, \"folds\": folds,\n               \"mean_macro_f1\": float(np.mean([f[\"macro_f1\"] for f in folds])),\n               \"std_macro_f1\": float(np.std([f[\"macro_f1\"] for f in folds])), \"oof_metrics\": metrics(y, pred),\n               \"mean_fit_seconds\": float(np.mean(fit_times)), \"mean_inference_seconds_per_record\": float(np.mean(infer_times)),\n               \"score_kind\": score_kind}\n        if spec[\"algorithm\"] == \"LogisticRegression\":\n            curve = []\n            for threshold in protocol[\"confidence_analysis\"][\"thresholds\"]:\n                kept = [i for i in ordered if scores[i] >= threshold]\n                curve.append({\"threshold\": threshold, \"accepted\": len(kept), \"coverage\": len(kept) / len(ordered),\n                              \"accepted_accuracy\": sum(predictions[i] == development[i][\"categoria\"] for i in kept) / len(kept) if kept else None,\n                              \"abstentions_per_category\": dict(Counter(development[i][\"categoria\"] for i in ordered if i not in set(kept)))})\n            row[\"uncalibrated_probability_coverage_analysis\"] = curve\n        rows.append(row)\n        oof_rows += [{\"candidate_id\": spec[\"id\"], \"record_id\": i, \"real\": development[i][\"categoria\"], \"predicted\": predictions[i],\n                      \"correct\": development[i][\"categoria\"] == predictions[i], \"score\": scores[i], \"score_kind\": score_kind} for i in ordered]\n    comparison = {\"experiment_version\": VERSION, \"phase\": \"development_only\", \"training_executed\": True, \"splits_created\": True, \"provenance\": provenance, \"candidates\": rows}\n    decision = choose_candidate(comparison, protocol)\n    selection = {\"experiment_version\": VERSION, \"provenance\": provenance, \"decision\": decision,\n                 \"comparison_sha256_canonical_json\": digest(comparison), \"final_test_used_for_selection\": False}\n    selection[\"selection_sha256_canonical_json\"] = digest(selection)\n    write_json(output / \"comparison.json\", comparison)\n    write_csv(output / \"development-oof.csv\", oof_rows)\n    write_csv(output / \"comparison.csv\", [{\"candidate\": r[\"id\"], \"cv_macro_f1_mean\": r[\"mean_macro_f1\"], \"cv_macro_f1_std\": r[\"std_macro_f1\"],\n              \"oof_accuracy\": r[\"oof_metrics\"][\"accuracy\"], \"oof_macro_f1\": r[\"oof_metrics\"][\"macro_f1\"], \"oof_weighted_f1\": r[\"oof_metrics\"][\"weighted_f1\"],\n              \"mean_fit_seconds\": r[\"mean_fit_seconds\"], \"mean_inference_seconds_per_record\": r[\"mean_inference_seconds_per_record\"]} for r in rows])\n    plot_comparison(rows, output)\n    write_json(output / \"selection.json\", selection)\n    print(json.dumps(decision, indent=2, ensure_ascii=False))\n    return selection\n\n\ndef plot_comparison(rows, output):\n    import matplotlib\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots(figsize=(9, 4.8))\n    ax.bar([r[\"id\"] for r in rows], [r[\"mean_macro_f1\"] for r in rows], yerr=[r[\"std_macro_f1\"] for r in rows], capsize=5)\n    ax.axhline(0.70, color=\"red\", linestyle=\"--\", label=\"Objetivo académico Macro F1 = 0.70\")\n    ax.set_ylim(0, 1.05)\n    ax.set_ylabel(\"Macro F1: media y desviación entre folds\")\n    ax.set_title(\"Desarrollo: CV de 4 folds por grupos (datos sintéticos)\")\n    ax.tick_params(axis=\"x\", labelrotation=15)\n    ax.legend(fontsize=8)\n    fig.tight_layout()\n    for suffix in [\"png\", \"pdf\"]:\n        fig.savefig(Path(output) / f\"comparison.{suffix}\", dpi=180)\n    plt.close(fig)\n\n\ndef error_rows(rows, predictions, scores, kind):\n    result = []\n    for row, prediction, score in zip(rows, predictions, scores):\n        correct = row[\"categoria\"] == prediction\n        reason = \"Acierto; comprobar evidencia textual con guía\" if correct else \"Posible confusión por vocabulario compartido o familia no vista; requiere revisión humana, no es una explicación causal\"\n        if not correct and row[\"is_boundary_case\"]:\n            reason = \"Caso fronterizo: posiblemente no distingue el componente afectado; revisar manualmente según guía, sin cambiar etiqueta de prueba\"\n        result.append({\"record_id\": row[\"record_id\"], \"titulo\": row[\"titulo\"], \"descripcion\": row[\"descripcion\"],\n                       \"real\": row[\"categoria\"], \"predicted\": prediction, \"correct\": correct, \"score\": score, \"score_kind\": kind,\n                       \"scenario_family_id\": row[\"scenario_family_id\"], \"leakage_group_id\": row[\"leakage_group_id\"],\n                       \"is_boundary_case\": row[\"is_boundary_case\"], \"possible_reason_to_review\": reason})\n    return result\n\n\ndef plot_matrix(report, output, normalized=False):\n    import matplotlib\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import ConfusionMatrixDisplay\n    import numpy as np\n    key = \"confusion_matrix_normalized\" if normalized else \"confusion_matrix\"\n    fig, ax = plt.subplots(figsize=(7.5, 6))\n    ConfusionMatrixDisplay(np.array(report[key]), display_labels=LABELS).plot(ax=ax, cmap=\"Blues\", values_format=\".2f\" if normalized else \"d\", colorbar=False)\n    ax.set_title(\"Prueba final sintética: \" + (\"normalizada por fila\" if normalized else \"conteos\"))\n    ax.set_xlabel(\"Categoría predicha\")\n    ax.set_ylabel(\"Categoría real\")\n    ax.tick_params(axis=\"x\", labelrotation=15)\n    fig.tight_layout()\n    for suffix in [\"png\", \"pdf\"]:\n        fig.savefig(Path(output) / f\"{key}.{suffix}\", dpi=180)\n    plt.close(fig)\n\n\ndef evaluate_final(records, protocol, splits, output, provenance):\n    \"\"\"Read frozen decision before touching final-test texts. Refitting uses dev only.\"\"\"\n    output = Path(output)\n    selection = read_json(output / \"selection.json\")\n    comparison = read_json(output / \"comparison.json\")\n    claimed = selection.pop(\"selection_sha256_canonical_json\")\n    prep.require(digest(selection) == claimed, \"SELECTION_CHANGED\")\n    selection[\"selection_sha256_canonical_json\"] = claimed\n    prep.require(selection[\"provenance\"] == provenance and digest(comparison) == selection[\"comparison_sha256_canonical_json\"], \"EXPERIMENT_CHANGED_AFTER_SELECTION\")\n    prep.require(selection[\"decision\"] == choose_candidate(comparison, protocol) and selection[\"final_test_used_for_selection\"] is False, \"INVALID_SELECTION\")\n    prep.require(not (output / \"final-test-started.json\").exists(), \"FINAL_TEST_ALREADY_OPENED: read existing results; do not retune or overwrite\")\n    # Exclusive creation prevents two workers from opening the test in the same run.\n    with (output / \"final-test-started.json\").open(\"x\", encoding=\"utf-8\") as file:\n        json.dump({\"opened_at_utc\": datetime.now(timezone.utc).isoformat(), \"selection_sha256\": claimed}, file)\n    by_id = {r[\"record_id\"]: r for r in records}\n    development = [by_id[i] for i in splits[\"development_ids\"]]\n    final = [by_id[i] for i in splits[\"final_test_ids\"]]\n    decision = selection[\"decision\"]\n    estimator = make_estimator(decision[\"candidate_spec\"], protocol)\n    estimator.fit([text_input(r) for r in development], [r[\"categoria\"] for r in development])\n    texts = [text_input(r) for r in final]\n    predictions = estimator.predict(texts).tolist()\n    scores, kind = score_outputs(estimator, texts, predictions, decision[\"candidate_spec\"])\n    y = [r[\"categoria\"] for r in final]\n    baseline_spec = protocol[\"candidates\"][0]\n    baseline = make_estimator(baseline_spec, protocol)\n    baseline.fit([text_input(r) for r in development], [r[\"categoria\"] for r in development])\n    baseline_report = metrics(y, baseline.predict(texts).tolist())\n    report = metrics(y, predictions)\n    final_gate = meets_gates(report[\"macro_f1\"], report[\"classification_report\"], baseline_report[\"macro_f1\"], protocol)\n    accepted = decision[\"cv_acceptance_passed\"] and final_gate\n    result = {\"experiment_version\": VERSION, \"phase\": \"final_test\", \"provenance\": provenance,\n              \"training_executed\": True, \"splits_created\": True,\n              \"selection_sha256_canonical_json\": claimed, \"candidate_id\": decision[\"candidate_id\"], \"score_kind\": kind,\n              \"metrics\": report, \"baseline_metrics\": baseline_report, \"cv_acceptance_passed\": decision[\"cv_acceptance_passed\"],\n              \"final_acceptance_passed\": final_gate, \"experimental_promotion_allowed\": accepted,\n              \"institutionally_validated\": False, \"model_exported\": False,\n              \"status\": \"EXPERIMENTAL_CANDIDATE_ACCEPTED\" if accepted else \"EXPERIMENT_COMPLETE_NOT_APPROVED_FOR_PROMOTION\"}\n    examples = error_rows(final, predictions, scores, kind)\n    # Preserve measurements before rendering; only final-metrics marks full success.\n    write_json(output / \"final-evaluation.json\", {\"result\": result, \"examples\": examples})\n    write_csv(output / \"final-examples.csv\", examples)\n    write_csv(output / \"final-errors.csv\", [e for e in examples if not e[\"correct\"]], fieldnames=list(examples[0]))\n    for key in [\"confusion_matrix\", \"confusion_matrix_normalized\"]:\n        write_csv(output / f\"{key}.csv\", [{\"real\": label, **dict(zip(LABELS, values))} for label, values in zip(LABELS, report[key])])\n    plot_matrix(report, output)\n    plot_matrix(report, output, normalized=True)\n    write_report(output, selection, comparison, result, examples)\n    write_json(output / \"final-metrics.json\", result)\n    print(json.dumps(result, indent=2, ensure_ascii=False))\n    return result\n\n\ndef write_report(output, selection, comparison, final, examples):\n    decision = selection[\"decision\"]\n    lines = [\"# Experimento 4.4 — resultados reales\", \"\", \"SYNTHETIC / DEMO / ACADEMIC DATASET. Sin validación institucional.\", \"\",\n             f\"Candidato elegido solo con CV: `{decision['candidate_id']}`. Estado: **{final['status']}**.\", \"\",\n             \"160 registros de desarrollo; 40 de prueba final, 10 por categoría. Grupos indivisibles; cuatro folds congelados. Solo título y descripción entran al clasificador.\", \"\",\n             \"## Comparación de desarrollo\", \"\", \"| Candidato | Macro F1 media | Desviación | Accuracy OOF |\", \"| --- | ---: | ---: | ---: |\"]\n    lines += [f\"| {r['id']} | {r['mean_macro_f1']:.4f} | {r['std_macro_f1']:.4f} | {r['oof_metrics']['accuracy']:.4f} |\" for r in comparison[\"candidates\"]]\n    m = final[\"metrics\"]\n    lines += [\"\", f\"Selección: {decision['reason']}. Candidatos cercanos: {', '.join(decision['near_tie_candidates'])}.\", \"\", \"## Prueba final\", \"\",\n              f\"Accuracy: {m['accuracy']:.4f}; Macro F1: {m['macro_f1']:.4f}; F1 ponderado: {m['weighted_f1']:.4f}.\", \"\",\n              \"| Categoría | Precision | Recall | F1 | Soporte |\", \"| --- | ---: | ---: | ---: | ---: |\"]\n    lines += [f\"| {label} | {m['classification_report'][label]['precision']:.4f} | {m['classification_report'][label]['recall']:.4f} | {m['classification_report'][label]['f1-score']:.4f} | {int(m['classification_report'][label]['support'])} |\" for label in LABELS]\n    lines += [\"\", f\"Criterios CV: {final['cv_acceptance_passed']}; criterios prueba: {final['final_acceptance_passed']}; promoción experimental: {final['experimental_promotion_allowed']}.\", \"\",\n              \"![Comparación CV](comparison.png)\", \"\", \"![Matriz de confusión](confusion_matrix.png)\", \"\", \"![Matriz normalizada](confusion_matrix_normalized.png)\", \"\",\n              \"## Aciertos y errores\", \"\", \"El CSV final-examples incluye los 40 textos, etiquetas, predicciones y scores; final-errors contiene los fallos. Las razones automáticas son hipótesis para revisar, no explicaciones causales.\", \"\"]\n    samples = [e for e in examples if not e[\"correct\"]][:4] + [e for e in examples if e[\"correct\"]][:4]\n    for example in samples:\n        lines += [f\"- {example['record_id']}: {example['real']} → {example['predicted']}; acierto={example['correct']}. {example['possible_reason_to_review']}\"]\n    lines += [\"\", \"## Limitaciones\", \"\", \"Corpus sintético pequeño con una sola fuente asistida y sin revisión humana independiente. Los 200 registros no son 200 escenarios independientes. Con 10 casos por clase, un error cambia recall en 0.10. La desviación entre folds no es un intervalo de confianza. Probabilidades logísticas sin calibrar; márgenes SVM no son probabilidades. No se fijó un umbral operativo; la curva de cobertura solo usa OOF de desarrollo. No modificar datos, grupos o parámetros tras observar la prueba. Un fallo de criterios debe documentarse antes de considerar 4.5.\", \"\",\n              \"No se exporta modelo ni se crean servicios; 4.5 requiere autorización. GitHub/Colab deben registrar por separado dónde se ejecutó el experimento.\", \"\"]\n    Path(output, \"REPORT.md\").write_text(\"\\n\".join(lines), encoding=\"utf-8\")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--phase\", choices=[\"freeze\", \"validate\", \"compare\", \"final\"], required=True)\n    parser.add_argument(\"--input-root\", type=Path, required=True)\n    parser.add_argument(\"--protocol\", type=Path, required=True)\n    parser.add_argument(\"--splits\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    parser.add_argument(\"--requirements\", type=Path, required=True)\n    parser.add_argument(\"--source-mode\", default=\"local_snapshot\")\n    args = parser.parse_args()\n    cache = args.output.resolve() / \".matplotlib-cache\"\n    cache.mkdir(parents=True, exist_ok=True)\n    os.environ[\"MPLCONFIGDIR\"] = str(cache)\n    records, dataset = prep.verify_dataset(prep.load_local_snapshot(args.input_root))\n    protocol = read_json(args.protocol)\n    if args.phase == \"freeze\":\n        prep.require(not args.splits.exists(), \"SPLITS_ALREADY_FROZEN\")\n        args.splits.parent.mkdir(parents=True, exist_ok=True)\n        write_json(args.splits, freeze_splits(records, protocol))\n        print(\"FROZEN_SPLITS_CREATED_BEFORE_TRAINING\")\n        return\n    splits = read_json(args.splits)\n    summary = validate_splits(records, protocol, splits)\n    if args.phase == \"validate\":\n        print(json.dumps(summary, ensure_ascii=False, indent=2))\n        return\n    requirements = args.requirements.read_text(encoding=\"utf-8\")\n    env = prep.verify_environment(sys.executable, requirements)\n    # Release-level 4.3 flags describe that immutable release, not this training run.\n    dataset_provenance = {key: value for key, value in dataset.items() if key not in {\"training_started\", \"splits_created\"}}\n    provenance = {\"dataset\": dataset_provenance, \"protocol_sha256_canonical_json\": digest(protocol), \"splits_sha256_canonical_json\": digest(splits),\n                  \"experiment_source_sha256_utf8_lf\": prep.canonical_hash(Path(__file__).read_bytes()),\n                  \"preparation_source_sha256_utf8_lf\": prep.canonical_hash(Path(prep.__file__).read_bytes()),\n                  \"requirements_sha256_utf8_lf\": prep.canonical_hash(requirements.encode()), \"environment\": env, \"source_mode\": args.source_mode,\n                  \"partition_summary\": summary}\n    if args.phase == \"compare\":\n        compare(records, protocol, splits, args.output, provenance)\n    else:\n        evaluate_final(records, protocol, splits, args.output, provenance)\n\n\nif __name__ == \"__main__\":\n    try:\n        main()\n    except Exception as error:\n        print(f\"EXPERIMENT_FAILED: {type(error).__name__}: {error}\", file=sys.stderr)\n        sys.exit(1)\n"
PROTOCOL_TEXT = "{\n  \"schema_version\": 1,\n  \"experiment_version\": \"ticket-category-experiment-v1.0.0\",\n  \"dataset_commit\": \"9d3103991911b1b06b9faca6dce779013dae3036\",\n  \"dataset_sha256\": \"2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69\",\n  \"seed\": 42,\n  \"input_columns\": [\"titulo\", \"descripcion\"],\n  \"text_join\": \"trim(title) + LF + trim(description)\",\n  \"holdout\": {\"method\": \"sha256-seeded-group-subset-dp-v1\", \"records\": 40, \"records_per_category\": 10},\n  \"validation\": {\"method\": \"StratifiedGroupKFold\", \"folds\": 4, \"shuffle\": true, \"group_column\": \"leakage_group_id\"},\n  \"vectorizer\": {\n    \"views\": [\"word\", \"char_wb\"],\n    \"word_ngram_range\": [1, 2],\n    \"char_ngram_range\": [3, 5],\n    \"lowercase\": true,\n    \"strip_accents\": \"unicode\",\n    \"sublinear_tf\": true,\n    \"min_df\": 1,\n    \"stop_words\": null,\n    \"view_weights\": [1.0, 1.0]\n  },\n  \"candidates\": [\n    {\"id\": \"dummy-majority\", \"algorithm\": \"DummyClassifier\", \"strategy\": \"most_frequent\"},\n    {\"id\": \"logreg-c1\", \"algorithm\": \"LogisticRegression\", \"C\": 1.0, \"solver\": \"lbfgs\", \"max_iter\": 2000},\n    {\"id\": \"logreg-c4\", \"algorithm\": \"LogisticRegression\", \"C\": 4.0, \"solver\": \"lbfgs\", \"max_iter\": 2000},\n    {\"id\": \"linear-svc-c1\", \"algorithm\": \"LinearSVC\", \"C\": 1.0, \"dual\": \"auto\", \"max_iter\": 10000},\n    {\"id\": \"linear-svc-c4\", \"algorithm\": \"LinearSVC\", \"C\": 4.0, \"dual\": \"auto\", \"max_iter\": 10000}\n  ],\n  \"selection\": {\n    \"primary\": \"mean_fold_macro_f1\",\n    \"recall_gate\": \"pooled_out_of_fold_per_category\",\n    \"minimum_macro_f1\": 0.70,\n    \"minimum_recall_per_category\": 0.60,\n    \"minimum_macro_f1_gain_over_baseline\": 0.10,\n    \"tie_tolerance\": \"maximum_population_std_of_best_and_candidate_fold_macro_f1\",\n    \"tie_preference\": [\"LogisticRegression\", \"LinearSVC\"],\n    \"within_algorithm_preference\": \"lowest_C\",\n    \"no_qualified_candidate\": \"evaluate_leading_candidate_once_but_do_not_promote\"\n  },\n  \"confidence_analysis\": {\n    \"source\": \"development_out_of_fold_only\",\n    \"probabilities\": \"uncalibrated_LogisticRegression_only\",\n    \"thresholds\": [0.25, 0.4, 0.55, 0.7, 0.85],\n    \"operational_threshold\": null\n  },\n  \"final_test\": {\"selected_candidate_only\": true, \"baseline_comparison\": true, \"same_acceptance_gates\": true, \"no_retuning\": true},\n  \"institutional_validation\": false,\n  \"persist_model\": false\n}\n"
SPLITS_TEXT = "{\n  \"schema_version\": 1,\n  \"experiment_version\": \"ticket-category-experiment-v1.0.0\",\n  \"dataset_sha256\": \"2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69\",\n  \"protocol_sha256_canonical_json\": \"c5eacbc7011971ad0f0ccbaf17b974866304386df5d7216869994a546fdb3959\",\n  \"seed\": 42,\n  \"method\": \"sha256-seeded-group-subset-dp-v1\",\n  \"development_ids\": [\n    \"ML-0011\",\n    \"ML-0012\",\n    \"ML-0013\",\n    \"ML-0014\",\n    \"ML-0015\",\n    \"ML-0016\",\n    \"ML-0017\",\n    \"ML-0018\",\n    \"ML-0019\",\n    \"ML-0020\",\n    \"ML-0021\",\n    \"ML-0022\",\n    \"ML-0023\",\n    \"ML-0024\",\n    \"ML-0025\",\n    \"ML-0026\",\n    \"ML-0027\",\n    \"ML-0028\",\n    \"ML-0029\",\n    \"ML-0030\",\n    \"ML-0031\",\n    \"ML-0032\",\n    \"ML-0033\",\n    \"ML-0034\",\n    \"ML-0035\",\n    \"ML-0036\",\n    \"ML-0037\",\n    \"ML-0038\",\n    \"ML-0039\",\n    \"ML-0040\",\n    \"ML-0041\",\n    \"ML-0042\",\n    \"ML-0043\",\n    \"ML-0044\",\n    \"ML-0045\",\n    \"ML-0046\",\n    \"ML-0047\",\n    \"ML-0048\",\n    \"ML-0049\",\n    \"ML-0050\",\n    \"ML-0051\",\n    \"ML-0052\",\n    \"ML-0053\",\n    \"ML-0054\",\n    \"ML-0055\",\n    \"ML-0061\",\n    \"ML-0062\",\n    \"ML-0063\",\n    \"ML-0064\",\n    \"ML-0065\",\n    \"ML-0066\",\n    \"ML-0067\",\n    \"ML-0068\",\n    \"ML-0069\",\n    \"ML-0070\",\n    \"ML-0071\",\n    \"ML-0072\",\n    \"ML-0073\",\n    \"ML-0074\",\n    \"ML-0075\",\n    \"ML-0076\",\n    \"ML-0077\",\n    \"ML-0078\",\n    \"ML-0079\",\n    \"ML-0080\",\n    \"ML-0081\",\n    \"ML-0082\",\n    \"ML-0083\",\n    \"ML-0084\",\n    \"ML-0085\",\n    \"ML-0091\",\n    \"ML-0092\",\n    \"ML-0093\",\n    \"ML-0094\",\n    \"ML-0095\",\n    \"ML-0096\",\n    \"ML-0097\",\n    \"ML-0098\",\n    \"ML-0099\",\n    \"ML-0100\",\n    \"ML-0101\",\n    \"ML-0102\",\n    \"ML-0103\",\n    \"ML-0104\",\n    \"ML-0105\",\n    \"ML-0106\",\n    \"ML-0107\",\n    \"ML-0108\",\n    \"ML-0109\",\n    \"ML-0110\",\n    \"ML-0116\",\n    \"ML-0117\",\n    \"ML-0118\",\n    \"ML-0119\",\n    \"ML-0120\",\n    \"ML-0121\",\n    \"ML-0122\",\n    \"ML-0123\",\n    \"ML-0124\",\n    \"ML-0125\",\n    \"ML-0131\",\n    \"ML-0132\",\n    \"ML-0133\",\n    \"ML-0134\",\n    \"ML-0135\",\n    \"ML-0136\",\n    \"ML-0137\",\n    \"ML-0138\",\n    \"ML-0139\",\n    \"ML-0140\",\n    \"ML-0141\",\n    \"ML-0142\",\n    \"ML-0143\",\n    \"ML-0144\",\n    \"ML-0145\",\n    \"ML-0146\",\n    \"ML-0147\",\n    \"ML-0148\",\n    \"ML-0149\",\n    \"ML-0150\",\n    \"ML-0151\",\n    \"ML-0152\",\n    \"ML-0153\",\n    \"ML-0154\",\n    \"ML-0155\",\n    \"ML-0156\",\n    \"ML-0157\",\n    \"ML-0158\",\n    \"ML-0159\",\n    \"ML-0160\",\n    \"ML-0161\",\n    \"ML-0162\",\n    \"ML-0163\",\n    \"ML-0164\",\n    \"ML-0165\",\n    \"ML-0171\",\n    \"ML-0172\",\n    \"ML-0173\",\n    \"ML-0174\",\n    \"ML-0175\",\n    \"ML-0176\",\n    \"ML-0177\",\n    \"ML-0178\",\n    \"ML-0179\",\n    \"ML-0180\",\n    \"ML-0181\",\n    \"ML-0182\",\n    \"ML-0183\",\n    \"ML-0184\",\n    \"ML-0185\",\n    \"ML-0191\",\n    \"ML-0192\",\n    \"ML-0193\",\n    \"ML-0194\",\n    \"ML-0195\",\n    \"ML-0196\",\n    \"ML-0197\",\n    \"ML-0198\",\n    \"ML-0199\",\n    \"ML-0200\"\n  ],\n  \"final_test_ids\": [\n    \"ML-0001\",\n    \"ML-0002\",\n    \"ML-0003\",\n    \"ML-0004\",\n    \"ML-0005\",\n    \"ML-0006\",\n    \"ML-0007\",\n    \"ML-0008\",\n    \"ML-0009\",\n    \"ML-0010\",\n    \"ML-0056\",\n    \"ML-0057\",\n    \"ML-0058\",\n    \"ML-0059\",\n    \"ML-0060\",\n    \"ML-0086\",\n    \"ML-0087\",\n    \"ML-0088\",\n    \"ML-0089\",\n    \"ML-0090\",\n    \"ML-0111\",\n    \"ML-0112\",\n    \"ML-0113\",\n    \"ML-0114\",\n    \"ML-0115\",\n    \"ML-0126\",\n    \"ML-0127\",\n    \"ML-0128\",\n    \"ML-0129\",\n    \"ML-0130\",\n    \"ML-0166\",\n    \"ML-0167\",\n    \"ML-0168\",\n    \"ML-0169\",\n    \"ML-0170\",\n    \"ML-0186\",\n    \"ML-0187\",\n    \"ML-0188\",\n    \"ML-0189\",\n    \"ML-0190\"\n  ],\n  \"folds\": [\n    {\n      \"fold\": 1,\n      \"training_ids\": [\n        \"ML-0011\",\n        \"ML-0012\",\n        \"ML-0013\",\n        \"ML-0014\",\n        \"ML-0015\",\n        \"ML-0016\",\n        \"ML-0017\",\n        \"ML-0018\",\n        \"ML-0019\",\n        \"ML-0020\",\n        \"ML-0021\",\n        \"ML-0022\",\n        \"ML-0023\",\n        \"ML-0024\",\n        \"ML-0025\",\n        \"ML-0031\",\n        \"ML-0032\",\n        \"ML-0033\",\n        \"ML-0034\",\n        \"ML-0035\",\n        \"ML-0041\",\n        \"ML-0042\",\n        \"ML-0043\",\n        \"ML-0044\",\n        \"ML-0045\",\n        \"ML-0046\",\n        \"ML-0047\",\n        \"ML-0048\",\n        \"ML-0049\",\n        \"ML-0050\",\n        \"ML-0051\",\n        \"ML-0052\",\n        \"ML-0053\",\n        \"ML-0054\",\n        \"ML-0055\",\n        \"ML-0061\",\n        \"ML-0062\",\n        \"ML-0063\",\n        \"ML-0064\",\n        \"ML-0065\",\n        \"ML-0066\",\n        \"ML-0067\",\n        \"ML-0068\",\n        \"ML-0069\",\n        \"ML-0070\",\n        \"ML-0071\",\n        \"ML-0072\",\n        \"ML-0073\",\n        \"ML-0074\",\n        \"ML-0075\",\n        \"ML-0076\",\n        \"ML-0077\",\n        \"ML-0078\",\n        \"ML-0079\",\n        \"ML-0080\",\n        \"ML-0091\",\n        \"ML-0092\",\n        \"ML-0093\",\n        \"ML-0094\",\n        \"ML-0095\",\n        \"ML-0101\",\n        \"ML-0102\",\n        \"ML-0103\",\n        \"ML-0104\",\n        \"ML-0105\",\n        \"ML-0106\",\n        \"ML-0107\",\n        \"ML-0108\",\n        \"ML-0109\",\n        \"ML-0110\",\n        \"ML-0116\",\n        \"ML-0117\",\n        \"ML-0118\",\n        \"ML-0119\",\n        \"ML-0120\",\n        \"ML-0121\",\n        \"ML-0122\",\n        \"ML-0123\",\n        \"ML-0124\",\n        \"ML-0125\",\n        \"ML-0141\",\n        \"ML-0142\",\n        \"ML-0143\",\n        \"ML-0144\",\n        \"ML-0145\",\n        \"ML-0146\",\n        \"ML-0147\",\n        \"ML-0148\",\n        \"ML-0149\",\n        \"ML-0150\",\n        \"ML-0151\",\n        \"ML-0152\",\n        \"ML-0153\",\n        \"ML-0154\",\n        \"ML-0155\",\n        \"ML-0161\",\n        \"ML-0162\",\n        \"ML-0163\",\n        \"ML-0164\",\n        \"ML-0165\",\n        \"ML-0171\",\n        \"ML-0172\",\n        \"ML-0173\",\n        \"ML-0174\",\n        \"ML-0175\",\n        \"ML-0176\",\n        \"ML-0177\",\n        \"ML-0178\",\n        \"ML-0179\",\n        \"ML-0180\",\n        \"ML-0191\",\n        \"ML-0192\",\n        \"ML-0193\",\n        \"ML-0194\",\n        \"ML-0195\",\n        \"ML-0196\",\n        \"ML-0197\",\n        \"ML-0198\",\n        \"ML-0199\",\n        \"ML-0200\"\n      ],\n      \"validation_ids\": [\n        \"ML-0026\",\n        \"ML-0027\",\n        \"ML-0028\",\n        \"ML-0029\",\n        \"ML-0030\",\n        \"ML-0036\",\n        \"ML-0037\",\n        \"ML-0038\",\n        \"ML-0039\",\n        \"ML-0040\",\n        \"ML-0081\",\n        \"ML-0082\",\n        \"ML-0083\",\n        \"ML-0084\",\n        \"ML-0085\",\n        \"ML-0096\",\n        \"ML-0097\",\n        \"ML-0098\",\n        \"ML-0099\",\n        \"ML-0100\",\n        \"ML-0131\",\n        \"ML-0132\",\n        \"ML-0133\",\n        \"ML-0134\",\n        \"ML-0135\",\n        \"ML-0136\",\n        \"ML-0137\",\n        \"ML-0138\",\n        \"ML-0139\",\n        \"ML-0140\",\n        \"ML-0156\",\n        \"ML-0157\",\n        \"ML-0158\",\n        \"ML-0159\",\n        \"ML-0160\",\n        \"ML-0181\",\n        \"ML-0182\",\n        \"ML-0183\",\n        \"ML-0184\",\n        \"ML-0185\"\n      ]\n    },\n    {\n      \"fold\": 2,\n      \"training_ids\": [\n        \"ML-0011\",\n        \"ML-0012\",\n        \"ML-0013\",\n        \"ML-0014\",\n        \"ML-0015\",\n        \"ML-0016\",\n        \"ML-0017\",\n        \"ML-0018\",\n        \"ML-0019\",\n        \"ML-0020\",\n        \"ML-0026\",\n        \"ML-0027\",\n        \"ML-0028\",\n        \"ML-0029\",\n        \"ML-0030\",\n        \"ML-0031\",\n        \"ML-0032\",\n        \"ML-0033\",\n        \"ML-0034\",\n        \"ML-0035\",\n        \"ML-0036\",\n        \"ML-0037\",\n        \"ML-0038\",\n        \"ML-0039\",\n        \"ML-0040\",\n        \"ML-0041\",\n        \"ML-0042\",\n        \"ML-0043\",\n        \"ML-0044\",\n        \"ML-0045\",\n        \"ML-0061\",\n        \"ML-0062\",\n        \"ML-0063\",\n        \"ML-0064\",\n        \"ML-0065\",\n        \"ML-0066\",\n        \"ML-0067\",\n        \"ML-0068\",\n        \"ML-0069\",\n        \"ML-0070\",\n        \"ML-0076\",\n        \"ML-0077\",\n        \"ML-0078\",\n        \"ML-0079\",\n        \"ML-0080\",\n        \"ML-0081\",\n        \"ML-0082\",\n        \"ML-0083\",\n        \"ML-0084\",\n        \"ML-0085\",\n        \"ML-0091\",\n        \"ML-0092\",\n        \"ML-0093\",\n        \"ML-0094\",\n        \"ML-0095\",\n        \"ML-0096\",\n        \"ML-0097\",\n        \"ML-0098\",\n        \"ML-0099\",\n        \"ML-0100\",\n        \"ML-0101\",\n        \"ML-0102\",\n        \"ML-0103\",\n        \"ML-0104\",\n        \"ML-0105\",\n        \"ML-0106\",\n        \"ML-0107\",\n        \"ML-0108\",\n        \"ML-0109\",\n        \"ML-0110\",\n        \"ML-0131\",\n        \"ML-0132\",\n        \"ML-0133\",\n        \"ML-0134\",\n        \"ML-0135\",\n        \"ML-0136\",\n        \"ML-0137\",\n        \"ML-0138\",\n        \"ML-0139\",\n        \"ML-0140\",\n        \"ML-0141\",\n        \"ML-0142\",\n        \"ML-0143\",\n        \"ML-0144\",\n        \"ML-0145\",\n        \"ML-0146\",\n        \"ML-0147\",\n        \"ML-0148\",\n        \"ML-0149\",\n        \"ML-0150\",\n        \"ML-0151\",\n        \"ML-0152\",\n        \"ML-0153\",\n        \"ML-0154\",\n        \"ML-0155\",\n        \"ML-0156\",\n        \"ML-0157\",\n        \"ML-0158\",\n        \"ML-0159\",\n        \"ML-0160\",\n        \"ML-0161\",\n        \"ML-0162\",\n        \"ML-0163\",\n        \"ML-0164\",\n        \"ML-0165\",\n        \"ML-0171\",\n        \"ML-0172\",\n        \"ML-0173\",\n        \"ML-0174\",\n        \"ML-0175\",\n        \"ML-0181\",\n        \"ML-0182\",\n        \"ML-0183\",\n        \"ML-0184\",\n        \"ML-0185\",\n        \"ML-0191\",\n        \"ML-0192\",\n        \"ML-0193\",\n        \"ML-0194\",\n        \"ML-0195\",\n        \"ML-0196\",\n        \"ML-0197\",\n        \"ML-0198\",\n        \"ML-0199\",\n        \"ML-0200\"\n      ],\n      \"validation_ids\": [\n        \"ML-0021\",\n        \"ML-0022\",\n        \"ML-0023\",\n        \"ML-0024\",\n        \"ML-0025\",\n        \"ML-0046\",\n        \"ML-0047\",\n        \"ML-0048\",\n        \"ML-0049\",\n        \"ML-0050\",\n        \"ML-0051\",\n        \"ML-0052\",\n        \"ML-0053\",\n        \"ML-0054\",\n        \"ML-0055\",\n        \"ML-0071\",\n        \"ML-0072\",\n        \"ML-0073\",\n        \"ML-0074\",\n        \"ML-0075\",\n        \"ML-0116\",\n        \"ML-0117\",\n        \"ML-0118\",\n        \"ML-0119\",\n        \"ML-0120\",\n        \"ML-0121\",\n        \"ML-0122\",\n        \"ML-0123\",\n        \"ML-0124\",\n        \"ML-0125\",\n        \"ML-0176\",\n        \"ML-0177\",\n        \"ML-0178\",\n        \"ML-0179\",\n        \"ML-0180\"\n      ]\n    },\n    {\n      \"fold\": 3,\n      \"training_ids\": [\n        \"ML-0021\",\n        \"ML-0022\",\n        \"ML-0023\",\n        \"ML-0024\",\n        \"ML-0025\",\n        \"ML-0026\",\n        \"ML-0027\",\n        \"ML-0028\",\n        \"ML-0029\",\n        \"ML-0030\",\n        \"ML-0031\",\n        \"ML-0032\",\n        \"ML-0033\",\n        \"ML-0034\",\n        \"ML-0035\",\n        \"ML-0036\",\n        \"ML-0037\",\n        \"ML-0038\",\n        \"ML-0039\",\n        \"ML-0040\",\n        \"ML-0041\",\n        \"ML-0042\",\n        \"ML-0043\",\n        \"ML-0044\",\n        \"ML-0045\",\n        \"ML-0046\",\n        \"ML-0047\",\n        \"ML-0048\",\n        \"ML-0049\",\n        \"ML-0050\",\n        \"ML-0051\",\n        \"ML-0052\",\n        \"ML-0053\",\n        \"ML-0054\",\n        \"ML-0055\",\n        \"ML-0061\",\n        \"ML-0062\",\n        \"ML-0063\",\n        \"ML-0064\",\n        \"ML-0065\",\n        \"ML-0071\",\n        \"ML-0072\",\n        \"ML-0073\",\n        \"ML-0074\",\n        \"ML-0075\",\n        \"ML-0076\",\n        \"ML-0077\",\n        \"ML-0078\",\n        \"ML-0079\",\n        \"ML-0080\",\n        \"ML-0081\",\n        \"ML-0082\",\n        \"ML-0083\",\n        \"ML-0084\",\n        \"ML-0085\",\n        \"ML-0096\",\n        \"ML-0097\",\n        \"ML-0098\",\n        \"ML-0099\",\n        \"ML-0100\",\n        \"ML-0106\",\n        \"ML-0107\",\n        \"ML-0108\",\n        \"ML-0109\",\n        \"ML-0110\",\n        \"ML-0116\",\n        \"ML-0117\",\n        \"ML-0118\",\n        \"ML-0119\",\n        \"ML-0120\",\n        \"ML-0121\",\n        \"ML-0122\",\n        \"ML-0123\",\n        \"ML-0124\",\n        \"ML-0125\",\n        \"ML-0131\",\n        \"ML-0132\",\n        \"ML-0133\",\n        \"ML-0134\",\n        \"ML-0135\",\n        \"ML-0136\",\n        \"ML-0137\",\n        \"ML-0138\",\n        \"ML-0139\",\n        \"ML-0140\",\n        \"ML-0151\",\n        \"ML-0152\",\n        \"ML-0153\",\n        \"ML-0154\",\n        \"ML-0155\",\n        \"ML-0156\",\n        \"ML-0157\",\n        \"ML-0158\",\n        \"ML-0159\",\n        \"ML-0160\",\n        \"ML-0161\",\n        \"ML-0162\",\n        \"ML-0163\",\n        \"ML-0164\",\n        \"ML-0165\",\n        \"ML-0176\",\n        \"ML-0177\",\n        \"ML-0178\",\n        \"ML-0179\",\n        \"ML-0180\",\n        \"ML-0181\",\n        \"ML-0182\",\n        \"ML-0183\",\n        \"ML-0184\",\n        \"ML-0185\",\n        \"ML-0191\",\n        \"ML-0192\",\n        \"ML-0193\",\n        \"ML-0194\",\n        \"ML-0195\",\n        \"ML-0196\",\n        \"ML-0197\",\n        \"ML-0198\",\n        \"ML-0199\",\n        \"ML-0200\"\n      ],\n      \"validation_ids\": [\n        \"ML-0011\",\n        \"ML-0012\",\n        \"ML-0013\",\n        \"ML-0014\",\n        \"ML-0015\",\n        \"ML-0016\",\n        \"ML-0017\",\n        \"ML-0018\",\n        \"ML-0019\",\n        \"ML-0020\",\n        \"ML-0066\",\n        \"ML-0067\",\n        \"ML-0068\",\n        \"ML-0069\",\n        \"ML-0070\",\n        \"ML-0091\",\n        \"ML-0092\",\n        \"ML-0093\",\n        \"ML-0094\",\n        \"ML-0095\",\n        \"ML-0101\",\n        \"ML-0102\",\n        \"ML-0103\",\n        \"ML-0104\",\n        \"ML-0105\",\n        \"ML-0141\",\n        \"ML-0142\",\n        \"ML-0143\",\n        \"ML-0144\",\n        \"ML-0145\",\n        \"ML-0146\",\n        \"ML-0147\",\n        \"ML-0148\",\n        \"ML-0149\",\n        \"ML-0150\",\n        \"ML-0171\",\n        \"ML-0172\",\n        \"ML-0173\",\n        \"ML-0174\",\n        \"ML-0175\"\n      ]\n    },\n    {\n      \"fold\": 4,\n      \"training_ids\": [\n        \"ML-0011\",\n        \"ML-0012\",\n        \"ML-0013\",\n        \"ML-0014\",\n        \"ML-0015\",\n        \"ML-0016\",\n        \"ML-0017\",\n        \"ML-0018\",\n        \"ML-0019\",\n        \"ML-0020\",\n        \"ML-0021\",\n        \"ML-0022\",\n        \"ML-0023\",\n        \"ML-0024\",\n        \"ML-0025\",\n        \"ML-0026\",\n        \"ML-0027\",\n        \"ML-0028\",\n        \"ML-0029\",\n        \"ML-0030\",\n        \"ML-0036\",\n        \"ML-0037\",\n        \"ML-0038\",\n        \"ML-0039\",\n        \"ML-0040\",\n        \"ML-0046\",\n        \"ML-0047\",\n        \"ML-0048\",\n        \"ML-0049\",\n        \"ML-0050\",\n        \"ML-0051\",\n        \"ML-0052\",\n        \"ML-0053\",\n        \"ML-0054\",\n        \"ML-0055\",\n        \"ML-0066\",\n        \"ML-0067\",\n        \"ML-0068\",\n        \"ML-0069\",\n        \"ML-0070\",\n        \"ML-0071\",\n        \"ML-0072\",\n        \"ML-0073\",\n        \"ML-0074\",\n        \"ML-0075\",\n        \"ML-0081\",\n        \"ML-0082\",\n        \"ML-0083\",\n        \"ML-0084\",\n        \"ML-0085\",\n        \"ML-0091\",\n        \"ML-0092\",\n        \"ML-0093\",\n        \"ML-0094\",\n        \"ML-0095\",\n        \"ML-0096\",\n        \"ML-0097\",\n        \"ML-0098\",\n        \"ML-0099\",\n        \"ML-0100\",\n        \"ML-0101\",\n        \"ML-0102\",\n        \"ML-0103\",\n        \"ML-0104\",\n        \"ML-0105\",\n        \"ML-0116\",\n        \"ML-0117\",\n        \"ML-0118\",\n        \"ML-0119\",\n        \"ML-0120\",\n        \"ML-0121\",\n        \"ML-0122\",\n        \"ML-0123\",\n        \"ML-0124\",\n        \"ML-0125\",\n        \"ML-0131\",\n        \"ML-0132\",\n        \"ML-0133\",\n        \"ML-0134\",\n        \"ML-0135\",\n        \"ML-0136\",\n        \"ML-0137\",\n        \"ML-0138\",\n        \"ML-0139\",\n        \"ML-0140\",\n        \"ML-0141\",\n        \"ML-0142\",\n        \"ML-0143\",\n        \"ML-0144\",\n        \"ML-0145\",\n        \"ML-0146\",\n        \"ML-0147\",\n        \"ML-0148\",\n        \"ML-0149\",\n        \"ML-0150\",\n        \"ML-0156\",\n        \"ML-0157\",\n        \"ML-0158\",\n        \"ML-0159\",\n        \"ML-0160\",\n        \"ML-0171\",\n        \"ML-0172\",\n        \"ML-0173\",\n        \"ML-0174\",\n        \"ML-0175\",\n        \"ML-0176\",\n        \"ML-0177\",\n        \"ML-0178\",\n        \"ML-0179\",\n        \"ML-0180\",\n        \"ML-0181\",\n        \"ML-0182\",\n        \"ML-0183\",\n        \"ML-0184\",\n        \"ML-0185\"\n      ],\n      \"validation_ids\": [\n        \"ML-0031\",\n        \"ML-0032\",\n        \"ML-0033\",\n        \"ML-0034\",\n        \"ML-0035\",\n        \"ML-0041\",\n        \"ML-0042\",\n        \"ML-0043\",\n        \"ML-0044\",\n        \"ML-0045\",\n        \"ML-0061\",\n        \"ML-0062\",\n        \"ML-0063\",\n        \"ML-0064\",\n        \"ML-0065\",\n        \"ML-0076\",\n        \"ML-0077\",\n        \"ML-0078\",\n        \"ML-0079\",\n        \"ML-0080\",\n        \"ML-0106\",\n        \"ML-0107\",\n        \"ML-0108\",\n        \"ML-0109\",\n        \"ML-0110\",\n        \"ML-0151\",\n        \"ML-0152\",\n        \"ML-0153\",\n        \"ML-0154\",\n        \"ML-0155\",\n        \"ML-0161\",\n        \"ML-0162\",\n        \"ML-0163\",\n        \"ML-0164\",\n        \"ML-0165\",\n        \"ML-0191\",\n        \"ML-0192\",\n        \"ML-0193\",\n        \"ML-0194\",\n        \"ML-0195\",\n        \"ML-0196\",\n        \"ML-0197\",\n        \"ML-0198\",\n        \"ML-0199\",\n        \"ML-0200\"\n      ]\n    }\n  ]\n}\n"
EXPERIMENT_SOURCE_SHA256 = "236d6c62a45b014eb74f5cf6b56ba305d948554a5aa4af87c75a0ae3eb9e37d0"
require(canonical_hash(EXPERIMENT_SOURCE.encode()) == EXPERIMENT_SOURCE_SHA256, "EXPERIMENT_SOURCE_MISMATCH")
source_directory = WORKSPACE / "sources"
input_directory = WORKSPACE / "inputs"
result_directory = WORKSPACE / "results"
source_directory.mkdir(parents=True, exist_ok=True)
for name, payload in payloads.items():
    destination = input_directory / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(payload)
preparation_text = "\"\"\"Stage 4.3 only: immutable inputs, safe loading, dataset and environment checks.\n\nStandard library only. No model fitting, predictions, split creation or database access.\nThis source is embedded into the notebook by build-colab-notebook.mjs.\n\"\"\"\nfrom collections import Counter\nfrom hashlib import sha256\nfrom io import BytesIO\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport stat\nimport subprocess\nimport sys\nfrom urllib.error import HTTPError, URLError\nfrom urllib.request import Request, build_opener, HTTPRedirectHandler\nimport zipfile\n\nREPOSITORY = \"C-David28/essalud-ticket-system\"\nDATASET_COMMIT = \"9d3103991911b1b06b9faca6dce779013dae3036\"\nDATASET_VERSION = \"tickets-category-dataset-v1.0.0\"\nPREPARATION_VERSION = \"stage-4.3-preparation-v1\"\nINSTALLER_VERSION = \"25.1.1\"\nDATASET_ROOT = \"ml/datasets/v1.0.0/\"\nSOURCE = \"SYNTHETIC / DEMO / ACADEMIC DATASET\"\nCATEGORIES = [\"SOPORTE\", \"REDES\", \"INFRAESTRUCTURA\", \"BIOMEDICO\"]\nEXPECTED_HASHES = {\n    DATASET_ROOT + \"incidents.jsonl\": \"2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69\",\n    DATASET_ROOT + \"families.json\": \"0f27342915a770acd019a31b7685c8a2eea8c2de7616a38ca2ac8c1f079078b7\",\n    \"ml/docs/LABELING-GUIDE.md\": \"7cb0ee07c57a6fb4a857bbde3ba8fc6f6b1e41d911d81682c62b98f316899cc9\",\n    DATASET_ROOT + \"manifest.json\": \"aa698ab7e4480e2a0f51085b5a5138b2267eb12a021eb08c590d58a7d11fdb61\",\n    \"apps/api/src/domain/ticket.ts\": \"8cfa93c9f1e9a8a7106c46118671d38c63cfce2041e514210430d0bf7560ce3a\",\n}\nALLOWED_FILES = (\n    DATASET_ROOT + \"incidents.jsonl\",\n    DATASET_ROOT + \"families.json\",\n    DATASET_ROOT + \"manifest.json\",\n    \"ml/docs/LABELING-GUIDE.md\",\n    \"apps/api/src/domain/ticket.ts\",\n)\nMAX_FILE_BYTES = 1_000_000\nMAX_ARCHIVE_BYTES = 3_000_000\n\n\ndef require(condition, code):\n    if not condition:\n        raise ValueError(code)\n\n\ndef canonical_hash(payload):\n    return sha256(payload.decode(\"utf-8\").replace(\"\\r\\n\", \"\\n\").encode(\"utf-8\")).hexdigest()\n\n\ndef parse_json(payload, code):\n    try:\n        return json.loads(payload.decode(\"utf-8\"))\n    except (ValueError, UnicodeError):\n        raise ValueError(code) from None\n\n\nclass NoRedirect(HTTPRedirectHandler):\n    # Never forward a private Authorization header to an unexpected host.\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        return None\n\n\ndef load_github(token=None, opener=None):\n    \"\"\"Read only the allowlisted files at a fixed SHA; token stays in memory/header.\"\"\"\n    request_opener = opener or build_opener(NoRedirect())\n    payloads = {}\n    for name in ALLOWED_FILES:\n        url = f\"https://api.github.com/repos/{REPOSITORY}/contents/{name}?ref={DATASET_COMMIT}\"\n        headers = {\"Accept\": \"application/vnd.github.raw+json\",\n                   \"X-GitHub-Api-Version\": \"2022-11-28\", \"User-Agent\": \"essalud-ml-stage-4.3\"}\n        if token:\n            headers[\"Authorization\"] = \"Bearer \" + token\n        try:\n            with request_opener.open(Request(url, headers=headers), timeout=30) as response:\n                payload = response.read(MAX_FILE_BYTES + 1)\n        except HTTPError as error:\n            raise RuntimeError(f\"GITHUB_READ_FAILED_HTTP_{error.code}: verify commit, access or use ZIP upload\") from None\n        except (URLError, OSError):\n            raise RuntimeError(\"GITHUB_READ_FAILED: network unavailable; use verified ZIP upload\") from None\n        require(len(payload) <= MAX_FILE_BYTES, \"GITHUB_FILE_TOO_LARGE\")\n        payloads[name] = payload\n    return payloads\n\n\ndef load_archive(payload):\n    \"\"\"Read an allowlisted git archive in memory. Never extract it to the filesystem.\"\"\"\n    require(isinstance(payload, bytes) and len(payload) <= MAX_ARCHIVE_BYTES, \"ARCHIVE_TOO_LARGE\")\n    result, seen, total = {}, set(), 0\n    try:\n        with zipfile.ZipFile(BytesIO(payload)) as archive:\n            require(len(archive.infolist()) <= 25, \"ARCHIVE_TOO_MANY_ENTRIES\")\n            for entry in archive.infolist():\n                name = entry.filename\n                original = entry.orig_filename\n                parts = PurePosixPath(name).parts\n                require(original == name and name and \"\\\\\" not in original and not name.startswith(\"/\") and\n                        \":\" not in name and \"..\" not in parts and name not in seen, \"UNSAFE_ARCHIVE_PATH\")\n                seen.add(name)\n                require(not stat.S_ISLNK(entry.external_attr >> 16), \"ARCHIVE_SYMLINK\")\n                if entry.is_dir():\n                    require(any(item.startswith(name) for item in ALLOWED_FILES), \"UNEXPECTED_ARCHIVE_DIRECTORY\")\n                    continue\n                require(name in ALLOWED_FILES, \"UNEXPECTED_ARCHIVE_FILE\")\n                total += entry.file_size\n                require(entry.file_size <= MAX_FILE_BYTES and total <= MAX_ARCHIVE_BYTES, \"ARCHIVE_EXPANDED_TOO_LARGE\")\n                result[name] = archive.read(entry)\n    except (zipfile.BadZipFile, RuntimeError):\n        raise ValueError(\"INVALID_OR_ENCRYPTED_ARCHIVE\") from None\n    require(set(result) == set(ALLOWED_FILES), \"ARCHIVE_MISSING_FILE\")\n    return result\n\n\ndef load_local_snapshot(repository_root):\n    \"\"\"Offline smoke tests only; same allowlist and hashes as GitHub/ZIP.\"\"\"\n    root = Path(repository_root)\n    return {name: (root / name).read_bytes() for name in ALLOWED_FILES}\n\n\ndef verify_dataset(payloads):\n    require(set(payloads) == set(ALLOWED_FILES), \"INPUT_FILE_SET_MISMATCH\")\n    require(all(isinstance(value, bytes) and len(value) <= MAX_FILE_BYTES for value in payloads.values()), \"INVALID_PAYLOAD\")\n    for name, digest in EXPECTED_HASHES.items():\n        require(canonical_hash(payloads[name]) == digest, \"IMMUTABLE_DATASET_HASH_MISMATCH\")\n    manifest = parse_json(payloads[DATASET_ROOT + \"manifest.json\"], \"INVALID_MANIFEST_JSON\")\n    require(isinstance(manifest, dict), \"INVALID_MANIFEST\")\n    require(manifest.get(\"dataset_version\") == DATASET_VERSION and manifest.get(\"schema_version\") == 1 and\n            manifest.get(\"source\") == SOURCE and manifest.get(\"hash_algorithm\") == \"SHA-256-UTF8-LF\", \"RELEASE_MISMATCH\")\n    require(manifest.get(\"categories\") == CATEGORIES and manifest.get(\"distribution\") == dict.fromkeys(CATEGORIES, 50), \"CATEGORY_MANIFEST_MISMATCH\")\n    require(manifest.get(\"record_count\") == 200 and manifest.get(\"family_count\") == 40 and\n            manifest.get(\"leakage_group_count\") == 37 and manifest.get(\"records_per_family\") == 5, \"COUNT_MANIFEST_MISMATCH\")\n    expected_manifest_hashes = {\"incidents.jsonl\": EXPECTED_HASHES[DATASET_ROOT + \"incidents.jsonl\"],\n                               \"families.json\": EXPECTED_HASHES[DATASET_ROOT + \"families.json\"],\n                               \"../../docs/LABELING-GUIDE.md\": EXPECTED_HASHES[\"ml/docs/LABELING-GUIDE.md\"]}\n    require(manifest.get(\"files\") == expected_manifest_hashes and manifest.get(\"training_status\") == \"NOT_STARTED\" and\n            manifest.get(\"splits_status\") == \"NOT_CREATED_STAGE_4_3\", \"FROZEN_MANIFEST_MISMATCH\")\n    match = re.search(r\"export const CATEGORIES\\s*=\\s*\\[([^\\]]+)\\]\", payloads[\"apps/api/src/domain/ticket.ts\"].decode(\"utf-8\"))\n    require(match is not None and re.findall(r\"['\\\"]([^'\\\"]+)['\\\"]\", match.group(1)) == CATEGORIES, \"DOMAIN_CATEGORY_MISMATCH\")\n    catalog = parse_json(payloads[DATASET_ROOT + \"families.json\"], \"INVALID_FAMILIES_JSON\")\n    require(isinstance(catalog, dict) and catalog.get(\"schema_version\") == 1 and\n            isinstance(catalog.get(\"families\"), list), \"INVALID_FAMILY_CATALOG\")\n    families = {row[\"scenario_family_id\"]: row for row in catalog[\"families\"]}\n    require(len(families) == len(catalog[\"families\"]) == 40, \"INVALID_FAMILY_COUNT\")\n    lines = payloads[DATASET_ROOT + \"incidents.jsonl\"].decode(\"utf-8\").splitlines()\n    records = [parse_json(line.encode(\"utf-8\"), \"INVALID_RECORD_JSON\") for line in lines]\n    require(len(records) == 200, \"INVALID_RECORD_COUNT\")\n    fields = {\"record_id\", \"titulo\", \"descripcion\", \"categoria\", \"scenario_family_id\", \"leakage_group_id\",\n              \"is_synthetic\", \"source\", \"dataset_version\", \"label_status\", \"label_rationale\",\n              \"writing_style\", \"length_band\", \"is_boundary_case\"}\n    ids, inputs = set(), set()\n    for row in records:\n        require(isinstance(row, dict) and set(row) == fields, \"INVALID_RECORD_FIELDS\")\n        require(isinstance(row[\"record_id\"], str) and re.fullmatch(r\"ML-\\d{4}\", row[\"record_id\"]) and\n                row[\"record_id\"] not in ids, \"INVALID_RECORD_ID\")\n        ids.add(row[\"record_id\"])\n        require(row[\"is_synthetic\"] is True and row[\"source\"] == SOURCE and row[\"dataset_version\"] == DATASET_VERSION and\n                row[\"label_status\"] == \"APPROVED\", \"INVALID_RECORD_PROVENANCE\")\n        require(row[\"categoria\"] in CATEGORIES, \"INVALID_CATEGORY\")\n        family = families.get(row[\"scenario_family_id\"])\n        require(family is not None and row[\"categoria\"] == family[\"categoria\"] and\n                row[\"leakage_group_id\"] == family[\"leakage_group_id\"], \"FAMILY_GROUP_MISMATCH\")\n        require(isinstance(row[\"titulo\"], str) and 5 <= len(row[\"titulo\"]) <= 200 and\n                isinstance(row[\"descripcion\"], str) and 10 <= len(row[\"descripcion\"]) <= 5000, \"INVALID_TEXT\")\n        text = (row[\"titulo\"].strip(), row[\"descripcion\"].strip())\n        require(text not in inputs, \"DUPLICATE_TEXT\")\n        inputs.add(text)\n    distribution = dict(Counter(row[\"categoria\"] for row in records))\n    require(distribution == dict.fromkeys(CATEGORIES, 50), \"UNBALANCED_DATASET\")\n    require(all(count == 5 for count in Counter(row[\"scenario_family_id\"] for row in records).values()), \"INVALID_FAMILY_SIZE\")\n    groups = len({row[\"leakage_group_id\"] for row in records})\n    require(groups == 37, \"INVALID_GROUP_COUNT\")\n    report = {\"dataset_version\": DATASET_VERSION, \"dataset_commit\": DATASET_COMMIT,\n              \"dataset_sha256\": EXPECTED_HASHES[DATASET_ROOT + \"incidents.jsonl\"],\n              \"records\": len(records), \"distribution\": distribution, \"families\": len(families),\n              \"leakage_groups\": groups, \"is_synthetic\": True, \"training_started\": False,\n              \"splits_created\": False, \"status\": \"DATASET_V1_VERIFIED\"}\n    return records, report\n\n\ndef parse_requirements(text):\n    pins = {}\n    for line in text.splitlines():\n        line = line.strip()\n        if not line or line.startswith(\"#\"):\n            continue\n        match = re.fullmatch(r\"([a-zA-Z0-9-]+)==([0-9]+(?:\\.[0-9]+)+(?:\\.post[0-9]+)?)\", line)\n        require(match is not None and match.group(1) not in pins, \"INVALID_DEPENDENCY_PIN\")\n        pins[match.group(1)] = match.group(2)\n    require(len(pins) == 18, \"INCOMPLETE_DEPENDENCY_PINS\")\n    return pins\n\n\ndef check_python():\n    require((3, 11) <= sys.version_info[:2] < (3, 14), \"PYTHON_UNSUPPORTED: use Python 3.11, 3.12 or 3.13\")\n\n\ndef prepare_environment(workspace, requirements_text, install=True):\n    \"\"\"Use a project-local venv so Colab's system packages remain untouched.\"\"\"\n    check_python()\n    parse_requirements(requirements_text)\n    workspace = Path(workspace).resolve()\n    workspace.mkdir(parents=True, exist_ok=True)\n    environment = workspace / \".venv\"\n    python = environment / (\"Scripts/python.exe\" if sys.platform == \"win32\" else \"bin/python\")\n    if not python.exists():\n        # Colab images may lack ensurepip; host pip manages this empty venv directly.\n        subprocess.run([sys.executable, \"-m\", \"venv\", \"--without-pip\", str(environment)], check=True, timeout=120)\n    requirements_file = workspace / \"requirements-stage-4.3.txt\"\n    requirements_file.write_text(requirements_text, encoding=\"utf-8\")\n    if install:\n        subprocess.run([sys.executable, \"-m\", \"pip\", \"--isolated\", \"--python\", str(python), \"install\", \"--disable-pip-version-check\",\n                        \"--only-binary=:all:\", \"--index-url\", \"https://pypi.org/simple\",\n                        f\"pip=={INSTALLER_VERSION}\", \"-r\", str(requirements_file)], check=True, timeout=600)\n    return python\n\n\ndef verify_environment(python, requirements_text):\n    pins = parse_requirements(requirements_text)\n    code = \"\"\"import importlib.metadata as m, json, sys\nimport numpy, scipy, pandas, sklearn, matplotlib\nprint(json.dumps({'python':sys.version.split()[0], 'pip_tool_version':m.version('pip'), 'packages':{name:m.version(name) for name in json.loads(sys.argv[1])}}))\n\"\"\"\n    result = subprocess.run([str(python), \"-c\", code, json.dumps(list(pins))],\n                            check=True, capture_output=True, text=True, timeout=120)\n    report = json.loads(result.stdout)\n    require(report[\"packages\"] == pins, \"INSTALLED_DEPENDENCY_VERSION_MISMATCH\")\n    require(report[\"pip_tool_version\"] == INSTALLER_VERSION, \"INSTALLER_VERSION_MISMATCH\")\n    require((3, 11) <= tuple(map(int, report[\"python\"].split(\".\")[:2])) < (3, 14), \"ENVIRONMENT_PYTHON_UNSUPPORTED\")\n    subprocess.run([str(python), \"-m\", \"pip\", \"check\"], check=True, timeout=60)\n    return report\n\n\ndef readiness_report(dataset_report, environment_report, requirements_text, source_mode, preparation_sha256):\n    require(re.fullmatch(r\"[0-9a-f]{64}\", preparation_sha256) is not None, \"INVALID_PREPARATION_HASH\")\n    return {\"stage\": \"4.3\", \"status\": \"READY_FOR_STAGE_4_4\", \"source_mode\": source_mode,\n            \"repository\": REPOSITORY, \"dataset\": dataset_report, \"environment\": environment_report,\n            \"preparation_version\": PREPARATION_VERSION, \"preparation_source_sha256_utf8_lf\": preparation_sha256,\n            \"requirements_sha256_utf8_lf\": canonical_hash(requirements_text.encode(\"utf-8\")),\n            \"training_started\": False, \"model_selected\": False, \"operational_database_access\": False}\n"
(source_directory / "preparation.py").write_text(preparation_text, encoding="utf-8")
(source_directory / "experiment.py").write_text(EXPERIMENT_SOURCE, encoding="utf-8")
(source_directory / "analyze_errors.py").write_text("\"\"\"Read-only postprocessing of measured OOF/final predictions; never fits a model.\"\"\"\nimport argparse\nfrom collections import Counter\nfrom hashlib import sha256\nfrom pathlib import Path\nimport csv\nimport json\nimport sys\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"colab\"))\nimport preparation as prep\n\n\ndef analyze(input_root, output, splits_path):\n    records, _ = prep.verify_dataset(prep.load_local_snapshot(input_root))\n    by_id = {r[\"record_id\"]: r for r in records}\n    output = Path(output)\n    comparison = json.loads((output / \"comparison.json\").read_text(encoding=\"utf-8\"))\n    selection = json.loads((output / \"selection.json\").read_text(encoding=\"utf-8\"))\n    final = json.loads((output / \"final-metrics.json\").read_text(encoding=\"utf-8\"))\n    splits = json.loads(Path(splits_path).read_text(encoding=\"utf-8\"))\n    split_hash = sha256(json.dumps(splits, ensure_ascii=False, sort_keys=True, separators=(\",\", \":\")).encode()).hexdigest()\n    prep.require(split_hash == final[\"provenance\"][\"splits_sha256_canonical_json\"], \"ANALYSIS_SPLITS_MISMATCH\")\n    candidate = selection[\"decision\"][\"candidate_id\"]\n    prep.require(final[\"candidate_id\"] == candidate, \"ANALYSIS_SELECTION_MISMATCH\")\n    with (output / \"development-oof.csv\").open(encoding=\"utf-8\", newline=\"\") as file:\n        measured = [r for r in csv.DictReader(file) if r[\"candidate_id\"] == candidate]\n    prep.require(len(measured) == 160 and len({r[\"record_id\"] for r in measured}) == 160, \"ANALYSIS_OOF_COVERAGE_MISMATCH\")\n    prep.require({r[\"record_id\"] for r in measured} == set(splits[\"development_ids\"]), \"ANALYSIS_OOF_IDS_MISMATCH\")\n    rows = []\n    for prediction in measured:\n        record = by_id[prediction[\"record_id\"]]\n        prep.require(prediction[\"real\"] == record[\"categoria\"], \"ANALYSIS_LABEL_MISMATCH\")\n        rows.append({**prediction, \"correct\": prediction[\"real\"] == prediction[\"predicted\"],\n                     \"titulo\": record[\"titulo\"], \"descripcion\": record[\"descripcion\"],\n                     \"scenario_family_id\": record[\"scenario_family_id\"], \"leakage_group_id\": record[\"leakage_group_id\"],\n                     \"is_boundary_case\": record[\"is_boundary_case\"], \"expected_label_rationale_for_review\": record[\"label_rationale\"]})\n    errors = [row for row in rows if not row[\"correct\"]]\n    for filename, values in [(\"development-examples.csv\", rows), (\"development-errors.csv\", errors)]:\n        with (output / filename).open(\"w\", encoding=\"utf-8\", newline=\"\") as file:\n            writer = csv.DictWriter(file, fieldnames=list(rows[0]))\n            writer.writeheader()\n            writer.writerows(values)\n    family_counts = Counter(r[\"scenario_family_id\"] for r in errors)\n    class_counts = Counter(r[\"real\"] for r in errors)\n    selected = next(r for r in comparison[\"candidates\"] if r[\"id\"] == candidate)\n    prep.require(abs((160 - len(errors)) / 160 - selected[\"oof_metrics\"][\"accuracy\"]) < 1e-12, \"ANALYSIS_MEASUREMENTS_MISMATCH\")\n    summary = {\"candidate_id\": candidate, \"development_records\": 160, \"development_errors\": len(errors),\n               \"errors_per_category\": dict(class_counts), \"errors_per_family\": dict(family_counts),\n               \"oof_accuracy\": selected[\"oof_metrics\"][\"accuracy\"], \"final_records\": 40,\n               \"final_errors\": int(round(40 * (1 - final[\"metrics\"][\"accuracy\"]))),\n               \"postprocessing_only\": True, \"analysis_source_sha256_utf8_lf\": prep.canonical_hash(Path(__file__).read_bytes())}\n    (output / \"error-analysis.json\").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + \"\\n\", encoding=\"utf-8\")\n    lines = [\"# Análisis de errores — predicciones observadas\", \"\", \"Datos sintéticos; no información institucional real.\", \"\",\n             f\"Candidato: `{candidate}`. Desarrollo OOF: {160-len(errors)}/160 aciertos, {len(errors)} fallos. Prueba final: {40-summary['final_errors']}/40 aciertos, {summary['final_errors']} fallos.\", \"\",\n             \"OOF significa que cada predicción de desarrollo proviene de un modelo que no vio ese grupo durante su ajuste. No se utiliza el modelo ajustado con todo desarrollo para medir aciertos sobre esos mismos textos.\", \"\",\n             \"## Distribución de fallos de desarrollo\", \"\", \"| Categoría real | Errores |\", \"| --- | ---: |\"]\n    lines += [f\"| {label} | {class_counts[label]} |\" for label in prep.CATEGORIES]\n    lines += [\"\", \"## Ejemplos de errores observados\", \"\"]\n    # Diverse families, not eight variants of the same concept.\n    samples, seen = [], set()\n    for row in errors:\n        if row[\"scenario_family_id\"] not in seen:\n            samples.append(row)\n            seen.add(row[\"scenario_family_id\"])\n        if len(samples) == 8:\n            break\n    for row in samples:\n        lines += [f\"### {row['record_id']}: {row['real']} → {row['predicted']}\", \"\",\n                  f\"**{row['titulo']}**. {row['descripcion']}\", \"\",\n                  \"Criterio de etiqueta esperado (solo para revisión, nunca input del clasificador): \" + row[\"expected_label_rationale_for_review\"], \"\"]\n    lines += [\"## Aciertos observados de desarrollo\", \"\"]\n    for label in prep.CATEGORIES:\n        example = next((r for r in rows if r[\"correct\"] and r[\"real\"] == label), None)\n        if example:\n            lines += [f\"- {example['record_id']} — {example['titulo']}: real={label}, predicción={example['predicted']}.\"]\n    lines += [\"\", \"## Interpretación para revisar\", \"\",\n              \"Las confusiones pueden reflejar vocabulario compartido (monitor, señal, cable, software, energía) y familias ausentes en el entrenamiento de un fold. Un vectorizador TF-IDF no comprende de forma fiable negaciones ni cuál es el componente afectado. Son hipótesis de revisión de los textos, no diagnósticos ni explicaciones causales demostradas de los pesos del modelo.\", \"\",\n              \"El resultado final perfecto no elimina los fallos OOF. Solo hay ocho grupos finales y un corpus sintético de una fuente asistida, sin revisión humana independiente. No añadir errores artificiales ni escoger otra prueba para bajar/subir métricas. Cualquier mejora posterior requiere una versión nueva y una evaluación declarada; no usar estos errores para retocar V1.\", \"\",\n              \"Los scores SVM son márgenes, incluso pueden ser negativos. No convertirlos a porcentaje ni interpretarlos como certeza. La política de abstención/umbral operativo queda pendiente de validación con desarrollo antes de integración. Este postprocesamiento no entrena, cambia la decisión ni exporta modelos.\", \"\"]\n    (output / \"ERROR-ANALYSIS.md\").write_text(\"\\n\".join(lines), encoding=\"utf-8\")\n    print(json.dumps(summary, ensure_ascii=False, indent=2))\n    return summary\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--input-root\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    parser.add_argument(\"--splits\", type=Path, required=True)\n    args = parser.parse_args()\n    analyze(args.input_root, args.output, args.splits)\n", encoding="utf-8")
protocol_path = source_directory / "protocol.json"
splits_path = source_directory / "splits.json"
protocol_path.write_text(PROTOCOL_TEXT, encoding="utf-8")
splits_path.write_text(SPLITS_TEXT, encoding="utf-8")
def run_phase(phase):
    subprocess.run([str(experiment_python), "-B", str(source_directory / "experiment.py"),
        "--phase", phase, "--input-root", str(input_directory), "--protocol", str(protocol_path),
        "--splits", str(splits_path), "--output", str(result_directory),
        "--requirements", str(WORKSPACE / "requirements-stage-4.3.txt"), "--source-mode", SOURCE_MODE], check=True)
def show_image(path):
    try:
        from IPython.display import display, Image
    except ImportError:
        print("Figura guardada:", path)
    else:
        display(Image(filename=str(path)))
run_phase("validate")


## 7. Entrenamiento y selección con desarrollo

Ejecuta cinco configuraciones en los mismos cuatro folds: 20 ajustes pequeños, sin GPU. Vocabulario/IDF se aprende dentro del pipeline. Guarda métricas por fold, OOF, tiempos y decisión con hashes.

La prueba final NO se abre aquí. Si ya existe selección, se muestra sin sobrescribir. Para reproducir desde cero usa un runtime nuevo y el mismo protocolo; no cambies la partición buscando un resultado favorable.


In [ ]:
RUN_COMPARISON = True
if RUN_COMPARISON:
    if not (result_directory / "selection.json").exists():
        run_phase("compare")
    selection = json.loads((result_directory / "selection.json").read_text(encoding="utf-8"))
    comparison = json.loads((result_directory / "comparison.json").read_text(encoding="utf-8"))
    summary = [{"candidate": row["id"], "cv_macro_f1_mean": row["mean_macro_f1"], "cv_macro_f1_std": row["std_macro_f1"],
        "oof_accuracy": row["oof_metrics"]["accuracy"],
        "oof_recall": {label: row["oof_metrics"]["classification_report"][label]["recall"] for label in CATEGORIES}}
        for row in comparison["candidates"]]
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(json.dumps(selection["decision"], ensure_ascii=False, indent=2))
    show_image(result_directory / "comparison.png")


## 8. Abrir prueba final una vez, después de congelar la decisión

Inicialmente **RUN_FINAL_TEST=False**. Revisa la selección de la celda anterior. Si quieres completar esta ejecución autorizada 4.4, cambia a **True** y ejecuta solo esta celda.

Se ajusta el candidato elegido con los 160 de desarrollo y se evalúa en los 40 reservados, con baseline de referencia. No se prueban otros candidatos en test ni se modifican parámetros. Un registro exclusivo evita reabrir/sobrescribir la prueba en el mismo directorio. Si falla luego de abrirse, conserva el error y consulta antes de reintentar.

Métricas reales, matrices de conteos/normalizada, aciertos y errores quedan en resultados. SVM entrega márgenes, NO porcentajes de confianza. Probabilidades logísticas, si corresponde, no están calibradas.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if not (result_directory / "final-metrics.json").exists():
        run_phase("final")
    final_report = json.loads((result_directory / "final-metrics.json").read_text(encoding="utf-8"))
    subprocess.run([str(experiment_python), "-B", str(source_directory / "analyze_errors.py"),
        "--input-root", str(input_directory), "--output", str(result_directory), "--splits", str(splits_path)], check=True)
    print(json.dumps(final_report, ensure_ascii=False, indent=2))
    show_image(result_directory / "confusion_matrix.png")
    show_image(result_directory / "confusion_matrix_normalized.png")
    try:
        from IPython.display import display, Markdown
    except ImportError:
        print("Reporte guardado:", result_directory / "REPORT.md")
    else:
        display(Markdown((result_directory / "REPORT.md").read_text(encoding="utf-8")))
        display(Markdown((result_directory / "ERROR-ANALYSIS.md").read_text(encoding="utf-8")))
else:
    print("FINAL_TEST_LOCKED: revisar decisión; activar RUN_FINAL_TEST solo en esta celda para completar la evaluación.")


## 9. Descargar y detenerse

Cambia **DOWNLOAD_RESULTS=True** después de la evaluación. Guarda el ZIP, matrices, comparación, métricas, CSV de aciertos/errores y reporte para el informe. La salida incluye trazabilidad de código/dataset/entorno; no incluye modelo ni secretos. El notebook del repositorio se guarda limpio, sin outputs.

Repite desde un runtime nuevo para comprobar reproducibilidad: mismas métricas/decisión/hash; los tiempos pueden variar. No adaptar el experimento tras observar el test. **Detente aquí: no iniciar 4.5.**


In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    require((result_directory / "final-metrics.json").exists(), "Completa primero la prueba final sin cambiar el protocolo")
    archive_path = WORKSPACE / "stage-4.4-results.zip"
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(result_directory.iterdir()):
            if path.is_file() and path.suffix in {".json", ".csv", ".md", ".png", ".pdf"}:
                archive.write(path, path.name)
        for name in ["protocol.json", "splits.json"]:
            archive.write(source_directory / name, name)
    from google.colab import files
    files.download(str(archive_path))
